# 1. Data Cleaning

## Objective

The objective of this step is to prepare the raw cardiac-failure data for reliable analysis.

We will:
- Load and inspect all available datasets
- Understand their structure and data types
- Identify the patient-level key used to connect datasets
- Assess missing values and duplicate records
- Validate categorical and numerical variables using the data dictionary
- Apply only justified cleaning and transformation rules
- Preserve clinically meaningful information wherever possible

The cleaned datasets will serve as the foundation for further analysis.


### 1. Import required libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import io

pd.set_option("display.max_columns", None)

### 2. Define the Data Location

In [164]:
DATA_PATH = Path(
     r"C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026"
)
#DATA_PATH=Path(".")
CLEANED_PATH = DATA_PATH / "cleaned_data"
CLEANED_PATH.mkdir(exist_ok=True)

print("Raw data folder:", DATA_PATH)
print("Cleaned data folder:", CLEANED_PATH)

Raw data folder: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026
Cleaned data folder: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data


### 3. Load the Datasets

In [165]:
demography = pd.read_csv(DATA_PATH / "demography.csv")
hospitalization = pd.read_csv(DATA_PATH / "hospitalization_discharge.csv")
cardiac = pd.read_csv(DATA_PATH / "cardiac_complications.csv")
labs = pd.read_csv(DATA_PATH / "labs.csv")
patient_prescriptions = pd.read_csv(DATA_PATH / "patient_precriptions.csv")
patient_history = pd.read_csv(DATA_PATH / "patienthistory.csv")
responsiveness = pd.read_csv(DATA_PATH / "responsivenes.csv")

data_dictionary = pd.read_excel(
    DATA_PATH / "Cardiac_failure_data_dictionary.xlsx"
)

print("All datasets loaded successfully.")

All datasets loaded successfully.


### 4. Understand each dataset
Before applying any cleaning rules, we first examine the structure of each dataset.

This helps us understand:
- the number of records and variables
- the relative size of each dataset
- whether datasets appear to contain one record per patient or multiple records per patient
- which datasets may require special handling before integration

This assessment will guide the cleaning and integration decisions that follow.

In [7]:
datasets = {
    "Demography": demography,
    "Hospitalization & Discharge": hospitalization,
    "Cardiac Complications": cardiac,
    "Labs": labs,
    "Patient Prescriptions": patient_prescriptions,
    "Patient History": patient_history,
    "Responsiveness": responsiveness
}

overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Unique Patients": df["inpatient_number"].nunique(),
        "Duplicate Patient IDs": df["inpatient_number"].duplicated().sum(),
        "Patients with Multiple Records": df["inpatient_number"].value_counts().gt(1).sum(),
        "One Row per Patient": df.shape[0] == df["inpatient_number"].nunique(),
    }
    for name, df in datasets.items()
])

overview

,Dataset,Rows,Columns,Unique Patients,Duplicate Patient IDs,Patients with Multiple Records,One Row per Patient
0,Demography,2009,7,2009,0,0,True
1,Hospitalization & Discharge,2008,21,2008,0,0,True
2,Cardiac Complications,2008,14,2008,0,0,True
3,Labs,2008,107,2008,0,0,True
4,Patient Prescriptions,15362,2,2007,13355,1998,False
5,Patient History,2008,17,2008,0,0,True
6,Responsiveness,2008,6,2008,0,0,True


### 5. Patient ID Consistency Check

The datasets were compared using `inpatient_number` to identify differences in patient coverage across sources.

The comparison identified:
- One patient ID present in the demographic dataset but absent from the hospitalization dataset.
- One hospitalization patient with no corresponding prescription record.

These records are retained at this stage because a mismatch or missing prescription record does not, by itself, indicate an invalid patient record.

The affected records will be investigated further before any removal or imputation decision is made.

In [8]:
master_ids = set(hospitalization["inpatient_number"])

for name, df in datasets.items():
    if name == "Patient Prescriptions":
        continue
    
    dataset_ids = set(df["inpatient_number"])
    
    missing_from_master = dataset_ids - master_ids
    missing_from_dataset = master_ids - dataset_ids
    
    if missing_from_master or missing_from_dataset:
        print(f"\n{name}")
        
        if missing_from_master:
            print("Present in dataset but not hospitalization:",
                  missing_from_master)
        
        if missing_from_dataset:
            print("Present in hospitalization but not dataset:",
                  missing_from_dataset)

missing_from_prescriptions = (
    master_ids - set(patient_prescriptions["inpatient_number"])
)

print("\nPatients with no prescription record:",
      missing_from_prescriptions)


Demography
Present in dataset but not hospitalization: {5}

Patients with no prescription record: {789308}


### Excluding Confirmed Invalid Patient Record

Patient ID 5 was identified as a data-entry/placeholder artifact based on its invalid ID format, absence from the other clinical datasets, and extensive missing demographic information.

In [9]:
demography = demography[
    demography["inpatient_number"] != 5
].reset_index(drop=True)

print("demography shape after exclusion:", demography.shape)
print("unique patients:", demography["inpatient_number"].nunique())

demography shape after exclusion: (2008, 7)
unique patients: 2008


#### Update the datasets dictionary
Because our datasets dictionary was created before we removed ID 5, update only the

In [10]:
datasets["Demography"] = demography

print(datasets["Demography"].shape)

(2008, 7)


### 6. Missing-Value Assessment

Missing values are assessed across all datasets before applying any cleaning rules.

The objective is to identify:
- which variables contain missing values
- the number and percentage of missing values
- datasets or variables with substantial missingness

No values are removed or imputed at this stage. Cleaning decisions will be based on the extent and context of the missing data.

### 6A. Detailed Missing-Value Summary

In [11]:
missing_summary = []

for name, df in datasets.items():
    for column in df.columns:
        missing_count = df[column].isna().sum()
        
        if missing_count > 0:
            missing_summary.append({
                "Dataset": name,
                "Column": column,
                "Missing Count": missing_count,
                "Missing %": round((missing_count / len(df)) * 100, 2)
            })

missing_summary = pd.DataFrame(missing_summary)

missing_summary = (
    missing_summary
    .sort_values(["Dataset", "Missing %"], ascending=[True, False])
    .reset_index(drop=True)
)

missing_summary

,Dataset,Column,Missing Count,Missing %
0,Cardiac Complications,tricuspid_valve_return_pressure,1826,90.94
1,Cardiac Complications,ea,1615,80.43
2,Cardiac Complications,mitral_valve_ams,1458,72.61
3,Cardiac Complications,lvef,1373,68.38
4,Cardiac Complications,tricuspid_valve_return_velocity,1218,60.66
...,...,...,...,...
111,Labs,sodium,11,0.55
112,Patient History,cci_score,5,0.25
113,Patient History,peptic_ulcer_disease,2,0.10
114,Patient History,moderate_to_severe_chronic_kidney_disease,2,0.10


### 6B. Missing Values by Dataset

In [12]:
missing_by_dataset = (
    missing_summary
    .groupby("Dataset")
    .agg(
        Columns_with_Missing=("Column", "count"),
        Total_Missing_Values=("Missing Count", "sum")
    )
    .reset_index()
)

missing_by_dataset

,Dataset,Columns_with_Missing,Total_Missing_Values
0,Cardiac Complications,7,9215
1,Demography,1,27
2,Hospitalization & Discharge,5,6149
3,Labs,99,48352
4,Patient History,4,10


### 6C. Variables With Highest Missingness

In [13]:
top_missing = (
    missing_summary
    .sort_values("Missing %", ascending=False)
    .head(20)
)

top_missing

,Dataset,Column,Missing Count,Missing %
13,Labs,cholinesterase,2008,100.00
8,Hospitalization & Discharge,respiratory_support,1966,97.91
9,Hospitalization & Discharge,time_of_death__days_from_admission,1964,97.81
14,Labs,homocysteine,1862,92.73
17,Labs,lipoprotein,1832,91.24
16,Labs,apolipoprotein_b,1832,91.24
15,Labs,apolipoprotein_a,1832,91.24
0,Cardiac Complications,tricuspid_valve_return_pressure,1826,90.94
18,Labs,erythrocyte_sedimentation_rate,1701,84.71
1,Cardiac Complications,ea,1615,80.43


## Conclusion — Phase 1: Cross-Dataset Structural Audit

Before cleaning any individual table, we established the structural facts that govern every cleaning decision made from this point forward:

- **Grain of each table** — 6 datasets are one row per patient; `patient_precriptions` is one-to-many (2,007 patients, 15,362 rows), so it cannot be merged or cleaned using the same logic as the rest.
- **Patient-ID integrity** — every table's population matches the 2,008-patient master set defined by `hospitalization`, with two exceptions: an orphan record (`inpatient_number = 5`) in `demography`, confirmed as a data artifact (wrong ID format, no data in any other table) and excluded; and patient `789308`, who is a valid patient with no prescription record — preserved, to be flagged later rather than treated as "zero medications."
- **Global missingness landscape** (Cells 6A–6C) — a ranked view across all 120 columns with missing data, which tells us *where* the largest gaps are before we look at any single table in isolation. This is kept as a standing reference, not repeated table-by-table.

These checks required comparing tables against each other and could only be done at the dataset level — that's why they came first.

## Phase 2 — Table-by-Table Cleaning

The cross-dataset audit established the overall data-quality landscape. The next phase focuses on cleaning and validating each dataset independently.

A dataset-level approach is used because each source has a different structure, variable meaning, level of measurement, and expected data quality. This allows missing values, data types, duplicates, and invalid values to be evaluated according to the purpose and definition of each variable rather than applying the same rule across all datasets.
The previously completed cross-dataset missingness summary remains as a reference so that decisions remain consistent across datasets.

### Demography — Step 1: Structure, Data Types & Duplicate Check

**Why this step:** Demography is cleaned first because we've already partially investigated it (the ID-5 exclusion) and it's the smallest table — a good one to confirm the table-by-table workflow on before moving to larger datasets like Labs. Before assessing missing values or invalid entries, we confirm each column's data type, take a visual sample of actual records, and check for two distinct duplication risks — identical full rows, and a repeated patient ID — since Demography is expected to be exactly one row per patient.

In [14]:
datasets["Demography"] = demography

print("Demography shape:", demography.shape)
print("\nData types:")
print(demography.dtypes)

print("\nSample records:")
display(demography.head())

print("\nFull-row duplicates:", demography.duplicated().sum())
print("Duplicate patient IDs:", demography["inpatient_number"].duplicated().sum())

Demography shape: (2008, 7)

Data types:
inpatient_number      int64
gender               object
weight              float64
height              float64
bmi                 float64
occupation           object
agecat               object
dtype: object

Sample records:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
0,827040,Female,50.0,1.45,23.781213,NaN,69-79
1,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
2,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
3,866418,Male,70.0,1.70,24.221453,farmer,59-69
4,775928,Male,65.0,1.70,22.491349,UrbanResident,69-79



Full-row duplicates: 0
Duplicate patient IDs: 0


### Demography — Step 2: Missing-Value Detail

**Why this step:** With the ID-5 artifact excluded and duplication ruled out, we look at missing values column-by-column — count, percentage, and data type together — to confirm exactly which fields still need attention before we move to value-level inspection.

In [15]:
demography_missing = pd.DataFrame({
    "Missing Count": demography.isna().sum(),
    "Missing %": (demography.isna().mean() * 100).round(2),
    "Data Type": demography.dtypes.astype(str)
})

demography_missing = (
    demography_missing[demography_missing["Missing Count"] > 0]
    .sort_values("Missing %", ascending=False)
)

demography_missing

,Missing Count,Missing %,Data Type
occupation,27,1.34,object


### Demography — Step 3: Value Inspection

**Why this step:** Before deciding how to treat any column, we look at the actual distribution of every field — including categorical ones — to catch impossible values (e.g., `weight = 0`), implausible ranges (e.g., `height`), and any unexpected category labels. No values are changed at this stage; this is investigation only, consistent with our detect-before-transform rule.

In [30]:
demography.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
inpatient_number,2008.0,NaN,NaN,NaN,797747.542829,41127.80174,722128.0,763164.5,798758.0,829399.75,905720.0
gender,2008,2,Female,1163,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,2008.0,NaN,NaN,NaN,52.483715,10.895935,0.0,45.0,50.0,60.0,115.0
height,2008.0,NaN,NaN,NaN,1.567869,0.096138,0.35,1.5,1.56,1.62,1.83
bmi,2008.0,NaN,NaN,NaN,21.791398,13.645009,0.0,18.491124,20.761246,23.4375,404.081633
occupation,1981,5,UrbanResident,1670,NaN,NaN,NaN,NaN,NaN,NaN,NaN
agecat,2008,8,69-79,715,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Demography — Step 4: Targeted Value Validation

The distribution review identified potentially invalid or implausible values in `weight`, `height`, and `bmi`, while categorical variables require validation of their actual category labels. We now investigate these findings at the record level before deciding whether any values require correction.
No assumptions or transformations are applied until the suspicious records are examined.

In [16]:
print("Weight = 0:")
display(demography[demography["weight"] == 0])

print("\nHeight outside plausible adult range (<1.2m or >2.2m):")
display(demography[(demography["height"] < 1.2) | (demography["height"] > 2.2)])

print("\nBMI = 0:")
display(demography[demography["bmi"] == 0])

print("\nBMI > 100:")
display(demography[demography["bmi"] > 100])

print("\nAll occupation categories:")
print(demography["occupation"].value_counts(dropna=False))

print("\nAll agecat categories:")
print(demography["agecat"].value_counts(dropna=False).sort_index())

Weight = 0:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
379,730511,Female,0.0,1.55,0.0,UrbanResident,79-89
388,785878,Male,0.0,1.65,0.0,UrbanResident,59-69
466,775572,Female,0.0,1.50,0.0,UrbanResident,69-79



Height outside plausible adult range (<1.2m or >2.2m):


,inpatient_number,gender,weight,height,bmi,occupation,agecat
317,837041,Female,49.0,0.48,212.673611,UrbanResident,69-79
320,815731,Male,49.5,0.35,404.081633,farmer,49-59
335,805044,Male,50.0,0.60,138.888889,UrbanResident,59-69
562,844739,Female,49.5,0.35,404.081633,UrbanResident,79-89



BMI = 0:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
379,730511,Female,0.0,1.55,0.0,UrbanResident,79-89
388,785878,Male,0.0,1.65,0.0,UrbanResident,59-69
466,775572,Female,0.0,1.50,0.0,UrbanResident,69-79



BMI > 100:


,inpatient_number,gender,weight,height,bmi,occupation,agecat
317,837041,Female,49.0,0.48,212.673611,UrbanResident,69-79
320,815731,Male,49.5,0.35,404.081633,farmer,49-59
335,805044,Male,50.0,0.60,138.888889,UrbanResident,59-69
562,844739,Female,49.5,0.35,404.081633,UrbanResident,79-89



All occupation categories:
occupation
UrbanResident    1670
farmer            198
Others             89
NaN                27
worker             17
Officer             7
Name: count, dtype: int64

All agecat categories:
agecat
21-29       4
29-39      12
39-49      56
49-59     106
59-69     368
69-79     715
79-89     646
89-110    101
Name: count, dtype: int64


### Demography — Step 5: Validate BMI Consistency

**Why this step:** BMI is mathematically derived from weight and height. Before modifying the identified BMI outliers, we verify whether the BMI values are internally consistent with the recorded weight and height. This helps determine whether the issue is with the BMI calculation itself or with one of its underlying measurements.

In [17]:
demography["calculated_bmi"] = (
    demography["weight"] / demography["height"]**2
)

bmi_check = demography[
    demography["bmi"].notna() &
    demography["calculated_bmi"].notna()
].copy()

bmi_check["bmi_difference"] = (
    bmi_check["bmi"] - bmi_check["calculated_bmi"]
).abs()

display(
    bmi_check.sort_values(
        "bmi_difference",
        ascending=False
    )[[
        "inpatient_number",
        "weight",
        "height",
        "bmi",
        "calculated_bmi",
        "bmi_difference"
    ]].head(10)
)

,inpatient_number,weight,height,bmi,calculated_bmi,bmi_difference
320,815731,49.5,0.35,404.081633,404.081633,2.842171e-13
562,844739,49.5,0.35,404.081633,404.081633,2.842171e-13
317,837041,49.0,0.48,212.673611,212.673611,1.136868e-13
335,805044,50.0,0.60,138.888889,138.888889,1.136868e-13
123,824845,53.0,1.55,22.060354,22.060354,5.329071e-14
377,840155,53.0,1.55,22.060354,22.060354,5.329071e-14
374,742244,57.0,1.35,31.275720,31.275720,5.329071e-14
1817,763848,53.0,1.55,22.060354,22.060354,5.329071e-14
1721,737932,40.0,1.45,19.024970,19.024970,4.973799e-14
921,779958,40.0,1.45,19.024970,19.024970,4.973799e-14


### Demography — Step 6: Data Dictionary Validation

**Why this step:** The BMI consistency check confirmed that the extreme BMI values are mathematically consistent with the recorded weight and height. We therefore investigate the variable definitions in the data dictionary before deciding how to treat the suspicious height and weight values.

In [34]:
print(data_dictionary.shape)
print(data_dictionary.columns.tolist())
display(data_dictionary.head(10))

(166, 3)
['Unnamed: 0', 'Variables', 'Description']


,Unnamed: 0,Variables,Description
0,1,inpatient_number,The patient unique ID
1,2,DestinationDischarge,"Destination of hospital discharge, recorded af..."
2,3,admission_ward,first admission ward
3,4,admission_way,possible ways of admission are Emergency vs. n...
4,5,occupation,occupation
5,6,discharge_department,the patient discharged from
6,7,visit_times,the number of hospital admissions before this ...
7,8,gender,"gender: Male, Female"
8,9,body_temperature,body temperature in degrees celsius
9,10,pulse,pulse rate (beats per minute)


### Demography — Step 7: Cleaning Invalid Measurements
**Finding:** Weight = 0 for 3 patients and height outside a documented adult-plausibility range (1.2–2.2m, per the data dictionary confirming height is measured in meters) for 4 patients. The BMI-consistency check confirmed BMI is mathematically derived from weight/height with no independent calculation error — so BMI is only ever wrong as a *consequence* of an invalid weight or height, never on its own.

**Cleaning decision:** Invalid weight and height values are treated as missing, not deleted — the patient records themselves are valid. BMI is recomputed only where both weight and height are valid, rather than trusting or individually nulling the original BMI column, since we've proven the two approaches are equivalent here but recomputation is self-correcting if that weren't the case.

In [18]:
HEIGHT_MIN_M = 1.2

# Identify invalid measurements
invalid_weight = demography["weight"] <= 0
invalid_height = demography["height"] < HEIGHT_MIN_M

# Treat invalid measurements as missing
demography.loc[invalid_weight, "weight"] = np.nan
demography.loc[invalid_height, "height"] = np.nan

# Recalculate BMI using valid weight and height
valid_bmi_inputs = (
    demography["weight"].notna() &
    demography["height"].notna()
)

demography.loc[valid_bmi_inputs, "bmi"] = (
    demography.loc[valid_bmi_inputs, "weight"]
    / demography.loc[valid_bmi_inputs, "height"] ** 2
)

# BMI cannot be calculated when either input is missing
demography.loc[~valid_bmi_inputs, "bmi"] = np.nan

print(
    "Invalid weights treated as missing:", invalid_weight.sum(),
    "| Invalid heights treated as missing:", invalid_height.sum()
)


Invalid weights treated as missing: 3 | Invalid heights treated as missing: 4


### Demography — Step 8: Standardize Categorical Columns

**Why this step:** `gender`, `occupation`, and `agecat` were confirmed clean (no stray casing/whitespace variants) during value inspection — no values need correction. This step only casts them to `category` dtype for consistency with the rest of the project, and renames `agecat` to `age_category` to match the underlying-word naming convention used elsewhere (`admission_ward`, `discharge_department`, etc.).

In [19]:
demography["gender"] = demography["gender"].astype("category")
demography["occupation"] = demography["occupation"].astype("category")

demography = demography.rename(columns={"agecat": "age_category"})
demography["age_category"] = demography["age_category"].astype("category")

datasets["Demography"] = demography
print(demography.dtypes)

inpatient_number       int64
gender              category
weight               float64
height               float64
bmi                  float64
occupation          category
age_category        category
calculated_bmi       float64
dtype: object


### Demography — Step 9: Data Type Validation — No Transformation Required

**Finding:** The categorical variables `gender`, `occupation`, and `age_category` are already stored as the `category` data type. The numeric variables also have appropriate numeric types for analysis.

**Decision:** No data type conversion is required. The existing data types are retained.

### Demography — Step 9: Occupation Category Standardization

**Why this step:** The occupation variable contains inconsistent capitalization and formatting across category labels, such as `UrbanResident`, `farmer`, `worker`, and `Officer`. Since these labels represent categorical values, inconsistent formatting can create unnecessary separate categories during grouping and analysis.

The labels are standardized for consistent presentation and aggregation without changing the underlying occupation category.

In [43]:
demography["occupation"].value_counts(dropna=False)

occupation
Urban Resident    1670
Farmer             198
Others              89
NaN                 27
Worker              17
Officer              7
Name: count, dtype: int64

In [20]:
occupation_mapping = {
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "Others": "Others",
    "worker": "Worker",
    "Officer": "Officer"
}

demography["occupation"] = (
    demography["occupation"]
    .map(occupation_mapping)
    .astype("category")
)

### Demography — Step 10: BMI Precision Standardization

**Why this step:** BMI is a derived continuous measurement and currently contains several decimal places due to mathematical calculation. Rounding BMI to two decimal places improves readability and avoids implying precision beyond what is meaningful for the measurement, while retaining sufficient precision for downstream analysis.

In [44]:
demography["bmi"] = demography["bmi"].round(2)

### Demography — Step 11: Post-Cleaning Validation

**Why this step:** After applying the cleaning and standardization rules, we validate the final dataset to confirm that the intended issues were resolved without changing the patient population or introducing new inconsistencies.

The validation checks patient-level uniqueness, remaining missing values, measurement validity, BMI consistency, categorical values, and final dataset structure before export.

In [21]:
print("Final shape:", demography.shape)
print("Unique patients:", demography["inpatient_number"].nunique())
print("Duplicate patient IDs:", demography["inpatient_number"].duplicated().sum())
print("Full-row duplicates:", demography.duplicated().sum())

print("\nInvalid weight values:")
print((demography["weight"] <= 0).sum())

print("\nInvalid height values:")
print((demography["height"] < 1.2).sum())

print("\nBMI outside 0–100:")
print(((demography["bmi"] <= 0) | (demography["bmi"] > 100)).sum())

print("\nMissing values:")
display(demography.isna().sum())

print("\nOccupation categories:")
print(demography["occupation"].value_counts(dropna=False))

print("\nData types:")
print(demography.dtypes)

Final shape: (2008, 8)
Unique patients: 2008
Duplicate patient IDs: 0
Full-row duplicates: 0

Invalid weight values:
0

Invalid height values:
0

BMI outside 0–100:
0

Missing values:


inpatient_number     0
gender               0
weight               3
height               4
bmi                  7
occupation          27
age_category         0
calculated_bmi       0
dtype: int64


Occupation categories:
occupation
Urban Resident    1670
Farmer             198
Others              89
NaN                 27
Worker              17
Officer              7
Name: count, dtype: int64

Data types:
inpatient_number       int64
gender              category
weight               float64
height               float64
bmi                  float64
occupation          category
age_category        category
calculated_bmi       float64
dtype: object


In [22]:
valid_bmi = (
    demography["weight"].notna() &
    demography["height"].notna() &
    demography["bmi"].notna()
)

calculated_bmi = (
    demography.loc[valid_bmi, "weight"]
    / demography.loc[valid_bmi, "height"] ** 2
)

bmi_difference = (
    demography.loc[valid_bmi, "bmi"] - calculated_bmi
).abs()

print("Maximum BMI rounding difference:", bmi_difference.max())

Maximum BMI rounding difference: 0.0


In [23]:
(demography["bmi"] <= 0) | (demography["bmi"] > 100)

0       False
1       False
2       False
3       False
4       False
        ...  
2003    False
2004    False
2005    False
2006    False
2007    False
Name: bmi, Length: 2008, dtype: bool

### Demography — Validation Conclusion

The Demography dataset passed all post-cleaning validation checks.

The dataset contains 2,008 unique patients with no duplicate patient IDs or duplicate rows. Invalid weight and height values identified during validation were treated as missing, and BMI was recalculated only where valid measurements were available.

Occupation labels were standardized, column naming was improved, and BMI was rounded to two decimal places. The final dataset contains no remaining invalid weight, height, or BMI values.

The cleaned Demography dataset is therefore ready for export and downstream analysis.

In [24]:
demography.to_csv(
    CLEANED_PATH / "demography_cleaned.csv",
    index=False
)

print("Saved:", CLEANED_PATH / "demography_cleaned.csv")

Saved: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\demography_cleaned.csv


In [25]:
demography_check = pd.read_csv(
    CLEANED_PATH / "demography_cleaned.csv"
)

print("Saved shape:", demography_check.shape)
print("Saved columns:", demography_check.columns.tolist())

Saved shape: (2008, 8)
Saved columns: ['inpatient_number', 'gender', 'weight', 'height', 'bmi', 'occupation', 'age_category', 'calculated_bmi']


In [26]:
dataset_overview = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Unique Patients": df["inpatient_number"].nunique(),
        "Duplicate Patient IDs": df["inpatient_number"].duplicated().sum(),
        "One Row per Patient": df.shape[0] == df["inpatient_number"].nunique()
    }
    for name, df in datasets.items()
])

dataset_overview

,Dataset,Rows,Columns,Unique Patients,Duplicate Patient IDs,One Row per Patient
0,Demography,2008,8,2008,0,True
1,Hospitalization & Discharge,2008,21,2008,0,True
2,Cardiac Complications,2008,14,2008,0,True
3,Labs,2008,107,2008,0,True
4,Patient Prescriptions,15362,2,2007,13355,False
5,Patient History,2008,17,2008,0,True
6,Responsiveness,2008,6,2008,0,True


## Hospitalization & Discharge — Step 1: Structure, Data Types & Duplicate Check

Hospitalization & Discharge contains information about the patient's hospital stay and outcomes. Before assessing missing values or validating individual variables, we first confirm the table's structure, data types, sample records, and whether patient IDs or complete rows are duplicated. This is especially important because the table may contain event-level information rather than exactly one record per patient.

In [27]:
print("Hospitalization & Discharge shape:", hospitalization.shape)

print("\nData types:")
print(hospitalization.dtypes)

print("\nSample records:")
display(hospitalization.head())

print("\nFull-row duplicates:", hospitalization.duplicated().sum())
print(
    "Duplicate patient IDs:",
    hospitalization["inpatient_number"].duplicated().sum()
)

Hospitalization & Discharge shape: (2008, 21)

Data types:
inpatient_number                                    int64
destinationdischarge                               object
admission_ward                                     object
admission_way                                      object
discharge_department                               object
visit_times                                         int64
respiratory_support                                object
oxygen_inhalation                                  object
dischargeday                                        int64
admission_date                                     object
outcome_during_hospitalization                     object
death_within_28_days                                int64
re_admission_within_28_days                         int64
death_within_3_months                               int64
re_admission_within_3_months                        int64
death_within_6_months                               int64
re_admission_

,inpatient_number,destinationdischarge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,dischargeday,admission_date,outcome_during_hospitalization,death_within_28_days,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24 00:00:00,Alive,0,0,0,0,0,0,NaN,NaN,0.0,NaN
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05 00:00:00,Alive,0,0,0,0,0,0,NaN,NaN,0.0,NaN
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18 00:00:00,Alive,0,0,0,0,0,0,NaN,NaN,0.0,NaN
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02 00:00:00,Alive,0,1,0,1,0,1,NaN,19.0,1.0,19.0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17 00:00:00,Alive,0,0,0,0,0,0,NaN,NaN,0.0,NaN



Full-row duplicates: 0
Duplicate patient IDs: 0


### Hospitalization & Discharge — Step 2: Missing-Value Detail

**Why this step:** The structure and patient-level uniqueness have been confirmed. We now examine missing values column-by-column to identify variables that require further investigation.

Missing values are not automatically treated as errors because some hospitalization variables describe events that may not occur for every patient. Event-related variables will therefore be investigated before any imputation or removal decision is made.

In [28]:
hospitalization_missing = pd.DataFrame({
    "Missing Count": hospitalization.isna().sum(),
    "Missing %": (hospitalization.isna().mean() * 100).round(2),
    "Data Type": hospitalization.dtypes.astype(str)
})

hospitalization_missing = (
    hospitalization_missing[
        hospitalization_missing["Missing Count"] > 0
    ]
    .sort_values("Missing %", ascending=False)
)

hospitalization_missing

,Missing Count,Missing %,Data Type
respiratory_support,1966,97.91,object
time_of_death__days_from_admission,1964,97.81,float64
time_to_emergency_department_within_6_months,1111,55.33,float64
readmission_time_days_from_admission,1107,55.13,float64
return_to_emergency_department_within_6_months,1,0.05,float64


### Hospitalization & Discharge — Step 3: Value Inspection

Before deciding how to treat any missing values, we look at the actual distributions — both categorical value counts and numeric ranges — to distinguish genuinely structural missingness (event never happened) from anything that looks like a data-quality problem. This is investigation only; no values are changed here.

In [29]:
categorical_cols = ["destinationdischarge", "admission_ward", "admission_way",
                     "discharge_department", "respiratory_support", "oxygen_inhalation",
                     "outcome_during_hospitalization"]

for col in categorical_cols:
    print(f"\n{col}:")
    print(hospitalization[col].value_counts(dropna=False))

print("\nNumeric ranges:")
display(hospitalization[["visit_times", "dischargeday", "time_of_death__days_from_admission",
                          "readmission_time_days_from_admission",
                          "time_to_emergency_department_within_6_months"]].describe())

print("\nadmission_date range:")
print(pd.to_datetime(hospitalization["admission_date"]).min(), "to",
      pd.to_datetime(hospitalization["admission_date"]).max())


destinationdischarge:
destinationdischarge
Home                  1344
HealthcareFacility     438
Unknown                212
Died                    14
Name: count, dtype: int64

admission_ward:
admission_ward
Cardiology     1547
GeneralWard     265
Others          181
ICU              15
Name: count, dtype: int64

admission_way:
admission_way
NonEmergency    1052
Emergency        956
Name: count, dtype: int64

discharge_department:
discharge_department
Cardiology     1703
GeneralWard     241
Others           52
ICU              12
Name: count, dtype: int64

respiratory_support:
respiratory_support
NaN     1966
IMV       25
NIMV      17
Name: count, dtype: int64

oxygen_inhalation:
oxygen_inhalation
OxygenTherapy    1898
AmbientAir        110
Name: count, dtype: int64

outcome_during_hospitalization:
outcome_during_hospitalization
Alive                    1890
DischargeAgainstOrder     107
Dead                       11
Name: count, dtype: int64

Numeric ranges:


,visit_times,dischargeday,time_of_death__days_from_admission,readmission_time_days_from_admission,time_to_emergency_department_within_6_months
count,2008.000000,2008.000000,44.000000,901.000000,897.000000
mean,1.092629,9.420817,29.522727,126.711432,126.734671
std,0.365946,8.030256,72.452226,145.025343,145.100169
min,1.000000,1.000000,1.000000,1.000000,1.000000
25%,1.000000,6.000000,2.000000,42.000000,42.000000
50%,1.000000,8.000000,4.500000,83.000000,83.000000
75%,1.000000,10.000000,18.750000,166.000000,166.000000
max,5.000000,123.000000,350.000000,2280.000000,2280.000000



admission_date range:
2016-01-01 00:00:00 to 2019-12-30 00:00:00


### Hospitalization & Discharge — Step 4: Cross-Field Consistency Validation

Several variables describe related hospitalization events from different perspectives. We therefore check whether these fields are logically consistent before making any cleaning decisions.
This helps distinguish legitimate event-dependent missingness from potential data-quality inconsistencies.

In [30]:
### Readmission consistency
readmit_contradictions = hospitalization[
    (hospitalization["re_admission_within_6_months"] == 0) &
    (hospitalization["readmission_time_days_from_admission"].notna())
]

display(readmit_contradictions[[
    "inpatient_number",
    "re_admission_within_28_days",
    "re_admission_within_3_months",
    "re_admission_within_6_months",
    "readmission_time_days_from_admission"
]])

,inpatient_number,re_admission_within_28_days,re_admission_within_3_months,re_admission_within_6_months,readmission_time_days_from_admission
30,743467,0,0,0,310.0
48,806646,0,0,0,463.0
71,778545,0,0,0,207.0
72,821564,0,0,0,190.0
84,782224,0,0,0,183.0
...,...,...,...,...,...
1975,801155,0,0,0,244.0
1982,739217,0,0,0,186.0
1985,776559,0,0,0,495.0
1992,729142,0,0,0,305.0


In [31]:
### Emergency department return consistency
ed_contradictions = hospitalization[
    (hospitalization["return_to_emergency_department_within_6_months"] == 0) &
    (hospitalization["time_to_emergency_department_within_6_months"].notna())
]

display(ed_contradictions[[
    "inpatient_number",
    "return_to_emergency_department_within_6_months",
    "time_to_emergency_department_within_6_months"
]])

,inpatient_number,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months
30,743467,0.0,310.0
48,806646,0.0,463.0
71,778545,0.0,207.0
84,782224,0.0,183.0
92,788057,0.0,192.0
...,...,...,...
1975,801155,0.0,244.0
1982,739217,0.0,186.0
1985,776559,0.0,495.0
1992,729142,0.0,305.0


In [32]:
### Outcome vs discharge destination consistency
outcome_destination_mismatch = hospitalization[
    (
        (hospitalization["outcome_during_hospitalization"] == "Alive") &
        (hospitalization["destinationdischarge"] == "Died")
    ) |
    (
        (hospitalization["outcome_during_hospitalization"] == "Dead") &
        (hospitalization["destinationdischarge"] != "Died")
    )
]

display(outcome_destination_mismatch[[
    "inpatient_number",
    "outcome_during_hospitalization",
    "destinationdischarge",
    "death_within_28_days",
    "death_within_3_months",
    "death_within_6_months",
    "time_of_death__days_from_admission"
]])

,inpatient_number,outcome_during_hospitalization,destinationdischarge,death_within_28_days,death_within_3_months,death_within_6_months,time_of_death__days_from_admission
288,728787,Alive,Died,0,0,0,NaN
486,768336,Alive,Died,0,0,0,NaN
1058,834443,Dead,Home,1,1,1,17.0
1391,828678,Dead,Unknown,1,1,1,3.0
1648,868748,Alive,Died,0,0,0,NaN
1781,753570,Alive,Died,0,0,0,NaN
1829,773886,Alive,Died,0,0,0,NaN


In [33]:
### Readmission window consistency
readmission_window_contradictions = hospitalization[
    (
        (hospitalization["re_admission_within_3_months"] == 1) &
        (hospitalization["re_admission_within_6_months"] == 0)
    )
]

display(readmission_window_contradictions[[
    "inpatient_number",
    "re_admission_within_28_days",
    "re_admission_within_3_months",
    "re_admission_within_6_months",
    "readmission_time_days_from_admission"
]])

,inpatient_number,re_admission_within_28_days,re_admission_within_3_months,re_admission_within_6_months,readmission_time_days_from_admission
517,741377,1,1,0,21.0
685,837990,1,1,0,15.0
844,741267,1,1,0,25.0


### Hospitalization & Discharge — Step 5: Data-Quality Flags from Cross-Field Validation

The cross-field validation identified inconsistencies between related outcome, readmission, emergency-return, and mortality variables. Before changing any values, we inspect the affected patient records and compare the related indicators and event-time variables.
The objective is to determine whether these differences represent data-entry inconsistencies, missing information, or differences in how the variables were recorded.

In [34]:
readmit_contradictions = hospitalization[
    (hospitalization["re_admission_within_6_months"] == 0) &
    (hospitalization["readmission_time_days_from_admission"] <= 180)
]
display(readmit_contradictions[["inpatient_number","re_admission_within_28_days","re_admission_within_3_months",
                                  "re_admission_within_6_months","readmission_time_days_from_admission"]])

ed_contradictions = hospitalization[
    (hospitalization["return_to_emergency_department_within_6_months"] == 0) &
    (hospitalization["time_to_emergency_department_within_6_months"] <= 180)
]
display(ed_contradictions[["inpatient_number","return_to_emergency_department_within_6_months",
                             "time_to_emergency_department_within_6_months"]])

,inpatient_number,re_admission_within_28_days,re_admission_within_3_months,re_admission_within_6_months,readmission_time_days_from_admission
132,730551,0,0,0,50.0
517,741377,1,1,0,21.0
600,740311,0,0,0,111.0
685,837990,1,1,0,15.0
844,741267,1,1,0,25.0


,inpatient_number,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months
183,775922,0.0,78.0
294,762948,0.0,121.0
885,800763,0.0,99.0
1360,851611,0.0,69.0
1493,764036,0.0,122.0
1701,772931,0.0,42.0


In [35]:
outcome_destination_mismatch = hospitalization[
    ((hospitalization["outcome_during_hospitalization"]=="Alive") & (hospitalization["destinationdischarge"]=="Died")) |
    ((hospitalization["outcome_during_hospitalization"]=="Dead") & (hospitalization["destinationdischarge"]!="Died"))
]
display(outcome_destination_mismatch[["inpatient_number","destinationdischarge","outcome_during_hospitalization",
    "death_within_28_days","death_within_3_months","death_within_6_months","time_of_death__days_from_admission"]])

,inpatient_number,destinationdischarge,outcome_during_hospitalization,death_within_28_days,death_within_3_months,death_within_6_months,time_of_death__days_from_admission
288,728787,Died,Alive,0,0,0,NaN
486,768336,Died,Alive,0,0,0,NaN
1058,834443,Home,Dead,1,1,1,17.0
1391,828678,Unknown,Dead,1,1,1,3.0
1648,868748,Died,Alive,0,0,0,NaN
1781,753570,Died,Alive,0,0,0,NaN
1829,773886,Died,Alive,0,0,0,NaN


In [36]:
death_time_missing = hospitalization[(hospitalization["death_within_6_months"]==1) & (hospitalization["time_of_death__days_from_admission"].isna())]
display(death_time_missing[["inpatient_number","outcome_during_hospitalization","destinationdischarge",
    "death_within_28_days","death_within_3_months","death_within_6_months","time_of_death__days_from_admission"]])

,inpatient_number,outcome_during_hospitalization,destinationdischarge,death_within_28_days,death_within_3_months,death_within_6_months,time_of_death__days_from_admission
11,748109,Alive,Home,0,0,1,NaN
15,852562,Alive,Home,0,0,1,NaN
21,765801,Alive,Home,0,0,1,NaN
24,814869,Alive,Home,0,0,1,NaN
32,783621,Alive,HealthcareFacility,0,0,1,NaN
70,804640,Alive,Home,0,0,1,NaN
104,747856,Alive,Unknown,0,0,1,NaN
114,860301,Alive,Home,0,0,1,NaN
131,787061,Alive,Home,0,0,1,NaN
132,730551,Alive,Home,0,1,1,NaN


### Investigation Findings

The cross-field investigation identified several apparent inconsistencies between event indicators and their corresponding event-time or outcome fields.

- A small number of patients have readmission indicators that are inconsistent across the 28-day, 3-month, and 6-month windows.
- Some patients have a recorded readmission or emergency-return time despite the corresponding 6-month indicator being 0.
- A small number of records show differences between discharge destination and hospitalization outcome.
- Some patients have a 6-month mortality indicator without a recorded time of death.

The available data dictionary does not provide sufficient detail to confidently determine the correct value for each contradictory record. Therefore, these fields are not arbitrarily overwritten or imputed. The original information is retained for downstream analysis, with event-dependent missingness preserved.

### Hospitalization & Discharge — Step 6: Cleaning Decisions

Based on the validation findings, we preserve event-dependent missing values and do not overwrite contradictory records without sufficient evidence. We standardize the date field and categorical values, then validate the cleaned dataset.

#### Hospitalization & Discharge — Step 6.1: Standardize Column Names

Column names are standardized to a consistent snake_case format to improve readability and simplify downstream analysis.

In [37]:
hospitalization = hospitalization.rename(columns={
    "destinationdischarge": "destination_discharge",
    "time_of_death__days_from_admission": "time_of_death_days_from_admission"
})

print(hospitalization.columns.tolist())


['inpatient_number', 'destination_discharge', 'admission_ward', 'admission_way', 'discharge_department', 'visit_times', 'respiratory_support', 'oxygen_inhalation', 'dischargeday', 'admission_date', 'outcome_during_hospitalization', 'death_within_28_days', 're_admission_within_28_days', 'death_within_3_months', 're_admission_within_3_months', 'death_within_6_months', 're_admission_within_6_months', 'time_of_death_days_from_admission', 'readmission_time_days_from_admission', 'return_to_emergency_department_within_6_months', 'time_to_emergency_department_within_6_months']


#### Hospitalization & Discharge — Step 6.2: Validate Admission Date Format

The admission date is checked for a meaningful time component before deciding whether to retain or remove it.

In [38]:
admission_datetime = pd.to_datetime(
    hospitalization["admission_date"],
    errors="coerce"
)

non_midnight = admission_datetime[
    (admission_datetime.dt.hour != 0) |
    (admission_datetime.dt.minute != 0) |
    (admission_datetime.dt.second != 0)
]

print("Total records:", len(admission_datetime))
print("Records with non-midnight time:", len(non_midnight))
print("Invalid/missing dates:", admission_datetime.isna().sum())

if len(non_midnight) > 0:
    display(non_midnight)

Total records: 2008
Records with non-midnight time: 0
Invalid/missing dates: 0


#### Cleaning Decision

All admission records have a midnight time component, so the time portion contains no additional information. The field is therefore retained as a date without the unnecessary time component.

In [39]:
hospitalization["admission_date"] = admission_datetime.dt.normalize()

print("Admission date type:", hospitalization["admission_date"].dtype)

Admission date type: datetime64[ns]


#### Hospitalization & Discharge — Step 6.3: Standardize Categorical Values

Categorical labels are standardized by removing extra spaces and improving readability of concatenated words while preserving their original meaning.

In [40]:
categorical_cols = [
    "destination_discharge",
    "admission_ward",
    "admission_way",
    "discharge_department",
    "respiratory_support",
    "oxygen_inhalation",
    "outcome_during_hospitalization"
]

# Remove leading/trailing spaces
for col in categorical_cols:
    hospitalization[col] = hospitalization[col].str.strip()

# Standardize concatenated categorical labels
category_mapping = {
    "destination_discharge": {
        "HealthcareFacility": "Healthcare Facility"
    },
    "admission_ward": {
        "GeneralWard": "General Ward"
    },
    "admission_way": {
        "NonEmergency": "Non Emergency"
    },
    "discharge_department": {
        "GeneralWard": "General Ward"
    },
    "oxygen_inhalation": {
        "OxygenTherapy": "Oxygen Therapy",
        "AmbientAir": "Ambient Air"
    },
    "outcome_during_hospitalization": {
        "DischargeAgainstOrder": "Discharge Against Order"
    }
}

for col, mapping in category_mapping.items():
    hospitalization[col] = hospitalization[col].replace(mapping)

In [41]:
for col in category_mapping:
    print(f"\n{col}:")
    print(hospitalization[col].value_counts(dropna=False))


destination_discharge:
destination_discharge
Home                   1344
Healthcare Facility     438
Unknown                 212
Died                     14
Name: count, dtype: int64

admission_ward:
admission_ward
Cardiology      1547
General Ward     265
Others           181
ICU               15
Name: count, dtype: int64

admission_way:
admission_way
Non Emergency    1052
Emergency         956
Name: count, dtype: int64

discharge_department:
discharge_department
Cardiology      1703
General Ward     241
Others            52
ICU               12
Name: count, dtype: int64

oxygen_inhalation:
oxygen_inhalation
Oxygen Therapy    1898
Ambient Air        110
Name: count, dtype: int64

outcome_during_hospitalization:
outcome_during_hospitalization
Alive                      1890
Discharge Against Order     107
Dead                         11
Name: count, dtype: int64


#### Hospitalization & Discharge — Step 6.4: Data-Quality Flags from Cross-Field Validation

In [42]:
readmit_contradictions = hospitalization[
    (hospitalization["re_admission_within_6_months"]==0) & (hospitalization["readmission_time_days_from_admission"]<=180)]
ed_contradictions = hospitalization[
    (hospitalization["return_to_emergency_department_within_6_months"]==0) & (hospitalization["time_to_emergency_department_within_6_months"]<=180)]
outcome_destination_mismatch = hospitalization[
    ((hospitalization["outcome_during_hospitalization"]=="Alive") & (hospitalization["destination_discharge"]=="Died")) |
    ((hospitalization["outcome_during_hospitalization"]=="Dead") & (hospitalization["destination_discharge"]!="Died"))]

event_time_missing = set()
for flag, timecol in [("death_within_6_months","time_of_death_days_from_admission"),
                        ("death_within_3_months","time_of_death_days_from_admission"),
                        ("re_admission_within_6_months","readmission_time_days_from_admission"),
                        ("return_to_emergency_department_within_6_months","time_to_emergency_department_within_6_months")]:
    event_time_missing |= set(hospitalization.loc[(hospitalization[flag]==1) & (hospitalization[timecol].isna()), "inpatient_number"])

hospitalization["outcome_destination_flag"] = hospitalization["inpatient_number"].isin(outcome_destination_mismatch["inpatient_number"])
hospitalization["event_time_missing_flag"] = hospitalization["inpatient_number"].isin(event_time_missing)
hospitalization["event_time_contradiction_flag"] = hospitalization["inpatient_number"].isin(
    set(readmit_contradictions["inpatient_number"]) | set(ed_contradictions["inpatient_number"]))

print(hospitalization.shape)
print(hospitalization[["outcome_destination_flag","event_time_missing_flag","event_time_contradiction_flag"]].sum())

(2008, 24)
outcome_destination_flag          7
event_time_missing_flag          24
event_time_contradiction_flag    11
dtype: int64


#### Hospitalization & Discharge — Step 6.5: Cleaning Validation

The cleaned dataset is validated to confirm that the intended transformations were applied and the patient-level structure was preserved.

In [43]:
print("Final shape:", hospitalization.shape)
print("Unique patients:", hospitalization["inpatient_number"].nunique())
print("Duplicate rows:", hospitalization.duplicated().sum())
print("Duplicate patient IDs:", hospitalization["inpatient_number"].duplicated().sum())

print("\nData types:")
print(hospitalization.dtypes)

print("\nMissing values:")
display(
    hospitalization.isna().sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print("\nStandardized categorical values:")
for col in categorical_cols:
    print(f"\n{col}:")
    print(hospitalization[col].value_counts(dropna=False))

Final shape: (2008, 24)
Unique patients: 2008
Duplicate rows: 0
Duplicate patient IDs: 0

Data types:
inpatient_number                                           int64
destination_discharge                                     object
admission_ward                                            object
admission_way                                             object
discharge_department                                      object
visit_times                                                int64
respiratory_support                                       object
oxygen_inhalation                                         object
dischargeday                                               int64
admission_date                                    datetime64[ns]
outcome_during_hospitalization                            object
death_within_28_days                                       int64
re_admission_within_28_days                                int64
death_within_3_months                                

respiratory_support                               1966
time_of_death_days_from_admission                 1964
time_to_emergency_department_within_6_months      1111
readmission_time_days_from_admission              1107
return_to_emergency_department_within_6_months       1
dtype: int64


Standardized categorical values:

destination_discharge:
destination_discharge
Home                   1344
Healthcare Facility     438
Unknown                 212
Died                     14
Name: count, dtype: int64

admission_ward:
admission_ward
Cardiology      1547
General Ward     265
Others           181
ICU               15
Name: count, dtype: int64

admission_way:
admission_way
Non Emergency    1052
Emergency         956
Name: count, dtype: int64

discharge_department:
discharge_department
Cardiology      1703
General Ward     241
Others            52
ICU               12
Name: count, dtype: int64

respiratory_support:
respiratory_support
NaN     1966
IMV       25
NIMV      17
Name: count, dtype: int64

oxygen_inhalation:
oxygen_inhalation
Oxygen Therapy    1898
Ambient Air        110
Name: count, dtype: int64

outcome_during_hospitalization:
outcome_during_hospitalization
Alive                      1890
Discharge Against Order     107
Dead                         11
Name: cou

In [44]:
assert hospitalization.shape == (2008, 24)
assert hospitalization["inpatient_number"].nunique() == 2008
assert hospitalization["inpatient_number"].duplicated().sum() == 0
assert hospitalization.duplicated().sum() == 0
assert hospitalization["admission_date"].notna().all()
assert all(c in hospitalization.columns for c in
           ["outcome_destination_flag","event_time_missing_flag","event_time_contradiction_flag"])

print("All structural validation checks passed.")

All structural validation checks passed.


### Hospitalization & Discharge — Step 7: Export Cleaned Dataset

The cleaned Hospitalization & Discharge dataset is exported as a separate CSV file so that it can be used consistently in the downstream descriptive, prescriptive, and predictive analyses.
The original raw dataset is preserved, while this cleaned version contains only the validated analytical fields and the required data-quality corrections.
**Expected outcome:**  
A finalized CSV file that is ready for analysis and can be reused across the remaining stages of the project without repeating the cleaning process.

In [45]:
# Define output file path
hospitalization_cleaned_path = (
    CLEANED_PATH / "hospitalization_discharge_cleaned.csv"
)

# Export cleaned dataset
hospitalization.to_csv(
    hospitalization_cleaned_path,
    index=False
)

print("Cleaned Hospitalization dataset exported successfully.")
print("File:", hospitalization_cleaned_path)

Cleaned Hospitalization dataset exported successfully.
File: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\hospitalization_discharge_cleaned.csv


In [46]:
# Confirm that the exported file exists
print("File exists:", hospitalization_cleaned_path.exists())

# Confirm exported dataset shape
hospitalization_exported = pd.read_csv(hospitalization_cleaned_path)

print("Exported shape:", hospitalization_exported.shape)
print("Exported columns:", hospitalization_exported.shape[1])
print("Exported unique patients:",
      hospitalization_exported["inpatient_number"].nunique())

File exists: True
Exported shape: (2008, 24)
Exported columns: 24
Exported unique patients: 2008


### Labs — Step 1: Structure, Data Types & Patient-Level Consistency

The Labs dataset is first assessed for its structure, data types, duplicate records, and patient-level consistency.

These checks establish whether the dataset has the expected analytical grain and whether its records can be reliably linked to the other patient-level datasets.

In [47]:
print("Dataset shape:", labs.shape)
print("Unique patients:", labs["inpatient_number"].nunique())

print("\nDuplicate rows:", labs.duplicated().sum())
print("Duplicate patient IDs:", labs["inpatient_number"].duplicated().sum())

print("\nData type summary:")
print(labs.dtypes.value_counts())

master_ids = set(hospitalization["inpatient_number"])
diff = set(labs["inpatient_number"]) ^ master_ids
print("\nPatient ID mismatch vs. master population:", diff if diff else "None")

print("\nSample records:")
display(labs.head())

datasets["Labs"] = labs

Dataset shape: (2008, 107)
Unique patients: 2008

Duplicate rows: 0
Duplicate patient IDs: 0

Data type summary:
float64    101
int64        6
Name: count, dtype: int64

Patient ID mismatch vs. master population: None

Sample records:


,inpatient_number,body_temperature,pulse,respiration,systolic_blood_pressure,diastolic_blood_pressure,map_value,fio2,creatinine_enzymatic_method,urea,uric_acid,glomerular_filtration_rate,cystatin,white_blood_cell,monocyte_ratio,monocyte_count,red_blood_cell,coefficient_of_variation_of_red_blood_cell_distribution_width,standard_deviation_of_red_blood_cell_distribution_width,mean_corpuscular_volume,hematocrit,lymphocyte_count,mean_hemoglobin_volume,mean_hemoglobin_concentration,mean_platelet_volume,basophil_ratio,basophil_count,eosinophil_ratio,eosinophil_count,hemoglobin,platelet,platelet_distribution_width,platelet_hematocrit,neutrophil_ratio,neutrophil_count,d_dimer,international_normalized_ratio,activated_partial_thromboplastin_time,thrombin_time,prothrombin_activity,prothrombin_time_ratio,fibrinogen,high_sensitivity_troponin,myoglobin,carbon_dioxide_binding_capacity,calcium,potassium,chloride,sodium,inorganic_phosphorus,serum_magnesium,creatine_kinase_isoenzyme_to_creatine_kinase,hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase,hydroxybutyrate_dehydrogenase,glutamic_oxaloacetic_transaminase,creatine_kinase,creatine_kinase_isoenzyme,lactate_dehydrogenase,brain_natriuretic_peptide,high_sensitivity_protein,nucleotidase,fucosidase,albumin,white_globulin_ratio,cholinesterase,glutamyltranspeptidase,glutamic_pyruvic_transaminase,glutamic_oxaliplatin,indirect_bilirubin,alkaline_phosphatase,globulin,direct_bilirubin,total_bilirubin,total_bile_acid,total_protein,erythrocyte_sedimentation_rate,cholesterol,low_density_lipoprotein_cholesterol,triglyceride,high_density_lipoprotein_cholesterol,homocysteine,apolipoprotein_a,apolipoprotein_b,lipoprotein,ph,standard_residual_base,standard_bicarbonate,partial_pressure_of_carbon_dioxide,total_carbon_dioxide,methemoglobin,hematocrit_blood_gas,reduced_hemoglobin,potassium_ion,chloride_ion,sodium_ion,glucose_blood_gas,lactate,measured_residual_base,measured_bicarbonate,carboxyhemoglobin,body_temperature_blood_gas,oxygen_saturation,partial_oxygen_pressure,oxyhemoglobin,anion_gap,free_calcium,total_hemoglobin
0,857781,36.7,87,19,102,64,76.666667,33,108.3,12.55,685.0,58.57,1.32,9.43,0.085,0.80,4.09,15.0,50.3,94.5,0.387,1.51,32.0,338.0,14.0,0.006,0.05,0.002,0.02,131.0,102.0,16.6,0.142,0.746,7.05,1.19,1.39,33.4,17.0,60.3,1.37,3.84,0.094,NaN,21.6,2.28,5.59,101.9,134.6,NaN,NaN,0.22,0.63,185.0,81.0,43.0,9.6,294.0,1500.17,7.6,3.5,20.5,38.1,1.6,NaN,83.0,65.0,1.02,11.4,61.0,23.8,6.9,18.3,4.9,61.9,NaN,3.46,1.90,2.69,0.84,NaN,NaN,NaN,NaN,7.44,-3.0,22.7,32.0,22.2,0.3,37.0,3.4,5.63,103.0,136.4,5.8,2.5,-2.1,21.2,0.4,37.0,97.0,93.0,95.9,17.8,1.14,125.0
1,743087,36.8,95,18,150,70,96.666667,33,62.0,4.29,170.0,85.43,1.25,5.32,0.066,0.35,3.93,13.0,40.7,88.6,0.349,1.18,28.9,326.0,11.3,0.002,0.01,0.009,0.05,114.0,123.0,16.1,0.139,0.702,3.73,1.06,1.16,35.9,18.4,65.1,1.18,2.62,0.017,NaN,26.5,2.28,3.62,105.6,144.0,NaN,NaN,0.16,0.84,170.0,21.0,107.0,17.2,203.0,361.70,NaN,2.2,21.9,40.7,1.6,NaN,17.0,19.0,NaN,13.1,69.0,25.4,5.2,18.3,2.4,66.1,NaN,3.15,1.26,0.84,1.30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,866418,36.5,98,18,102,67,78.666667,33,185.1,15.99,567.0,31.51,2.43,13.01,0.051,0.67,4.43,12.9,44.5,96.4,0.427,0.75,32.6,339.0,12.0,0.002,0.03,0.002,0.02,144.0,169.0,16.5,0.203,0.887,11.54,0.83,1.10,36.2,14.9,85.7,1.09,5.74,0.010,NaN,21.6,2.56,4.15,107.0,142.2,NaN,NaN,0.38,0.77,105.0,9.0,38.0,14.6,137.0,293.95,NaN,2.3,12.3,33.1,1.2,NaN,27.0,10.0,1.10,7.5,76.0,27.7,4.4,11.9,2.4,60.8,NaN,3.79,2.13,1.53,0.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,775928,36.0,73,19,110,74,86.000000,33,104.8,8.16,635.0,58.01,2.32,2.18,0.087,0.19,2.71,18.8,69.0,102.2,0.277,0.71,34.7,339.0,12.4,0.000,0.00,0.028,0.06,94.0,35.0,19.4,0.032,0.559,1.22,1.39,1.37,38.3,18.3,60.9,1.36,2.63,0.349,NaN,21.1,2.35,3.76,101.6,136.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1071.40,NaN,11.2,19.4,40.1,1.5,NaN,134.0,9.0,NaN,26.2

#### Step 1 — Conclusion
The Labs dataset contains 2,008 patient-level records and 107 variables. Each patient appears once, with no duplicate records or duplicate patient IDs.
All Labs patient IDs match the master hospitalization population, confirming that the dataset can be reliably linked with the other patient-level datasets.

### Labs — Step 2: Missingness Profile & Value Inspection

The Labs dataset contains a large number of clinical measurements, many of which may be collected only for specific tests or clinical situations.
Missingness and basic value distributions are therefore reviewed together to distinguish potentially event-dependent missing values from variables that may require cleaning.
This step is used to identify actionable data-quality issues rather than to treat missing values as errors by default.

In [48]:
labs_missing = pd.DataFrame({
    "Missing Count": labs.isna().sum(),
    "Missing %": (labs.isna().mean() * 100).round(2),
    "Data Type": labs.dtypes.astype(str)
})
labs_missing = labs_missing[labs_missing["Missing Count"] > 0].sort_values("Missing %", ascending=False)

print(f"Columns with missing values: {len(labs_missing)} of {labs.shape[1]}")
display(labs_missing.head(20))

# Automated scan: which numeric columns contain zero or negative values?
numeric_cols = labs.select_dtypes(include="number").columns.drop("inpatient_number")
zero_or_negative = pd.DataFrame({
    "Zero Count": (labs[numeric_cols] == 0).sum(),
    "Negative Count": (labs[numeric_cols] < 0).sum(),
})
zero_or_negative = zero_or_negative[(zero_or_negative["Zero Count"] > 0) | (zero_or_negative["Negative Count"] > 0)]
zero_or_negative = zero_or_negative.sort_values("Zero Count", ascending=False)

print(f"\nColumns containing zero or negative values: {len(zero_or_negative)}")
display(zero_or_negative)

Columns with missing values: 99 of 107


,Missing Count,Missing %,Data Type
cholinesterase,2008,100.00,float64
homocysteine,1862,92.73,float64
apolipoprotein_a,1832,91.24,float64
apolipoprotein_b,1832,91.24,float64
lipoprotein,1832,91.24,float64
erythrocyte_sedimentation_rate,1701,84.71,float64
myoglobin,1610,80.18,float64
serum_magnesium,1601,79.73,float64
inorganic_phosphorus,1601,79.73,float64
glutamic_oxaliplatin,1416,70.52,float64



Columns containing zero or negative values: 17


,Zero Count,Negative Count
eosinophil_count,202,0
eosinophil_ratio,198,0
methemoglobin,87,0
basophil_count,54,0
high_sensitivity_troponin,51,0
carboxyhemoglobin,36,0
basophil_ratio,29,0
measured_residual_base,13,678
standard_residual_base,8,708
d_dimer,4,0


#### Step 2 — Conclusion

The missingness profile shows substantial variation across laboratory variables, including completely missing and highly sparse clinical measurements. Basic value inspection also identified zero values in several variables.
Not all zero or negative values represent errors. Zero values in laboratory measurements may be legitimate results, while negative residual-base values can represent valid blood-gas measurements.
However, zero values in routine vital signs such as blood pressure, pulse, and respiration are not meaningful measurements and will be addressed as a targeted cleaning decision in Step 4.
The missingness patterns and value profile provide the evidence for the targeted cleaning decisions in the following steps.

### Labs — Step 3: Missingness-Based Cleaning Decisions

Missingness is not automatically treated as an error because some clinical tests are performed only for specific situations.

`cholinesterase` is completely missing and therefore provides no patient-level information in this dataset, so it is removed.

The blood-gas variables show substantial and consistent missingness across the panel. Rather than imputing these values or removing the variables, the original missingness is retained. A panel-level flag is also created to identify patients with at least one recorded blood-gas measurement, preserving the distinction between observed and unobserved testing for downstream analysis.

In [49]:
completely_missing = labs.columns[labs.isna().all()].tolist()
print("Completely missing variables:", completely_missing)

labs = labs.drop(columns=completely_missing)

blood_gas_cols = ["ph", "standard_residual_base", "standard_bicarbonate", "partial_pressure_of_carbon_dioxide",
    "total_carbon_dioxide", "methemoglobin", "hematocrit_blood_gas", "reduced_hemoglobin", "potassium_ion",
    "chloride_ion", "sodium_ion", "glucose_blood_gas", "lactate", "measured_residual_base", "measured_bicarbonate",
    "carboxyhemoglobin", "body_temperature_blood_gas", "oxygen_saturation", "partial_oxygen_pressure",
    "oxyhemoglobin", "free_calcium", "total_hemoglobin"]

labs["blood_gas_panel_observed"] = labs[blood_gas_cols].notna().any(axis=1)

datasets["Labs"] = labs

print("Updated Labs shape:", labs.shape)
print("\nBlood-gas panel indicator:")
print(labs["blood_gas_panel_observed"].value_counts())


Completely missing variables: ['cholinesterase']
Updated Labs shape: (2008, 107)

Blood-gas panel indicator:
blood_gas_panel_observed
False    1015
True      993
Name: count, dtype: int64


### Step 3 — Conclusion

`cholinesterase` was removed because it contained no observed values. Sparse blood-gas measurements were retained without imputation, and a panel-level indicator was created to identify patients with at least one recorded blood-gas measurement.

The indicator shows that 993 patients have observed blood-gas measurements, while 1,015 do not. This preserves the original clinical information while making the panel-level availability directly usable in downstream analysis.

  ### Labs — Step 4: Invalid Values

**Supporting Argument:** Step 2 identified zero or negative values across several laboratory variables. Many of these values are clinically valid — for example, eosinophil and basophil measurements can be zero, while residual-base and anion-gap measurements can be negative.

The remaining five variables — `systolic_blood_pressure`, `diastolic_blood_pressure`, `map_value`, `pulse`, and `respiration` — contain zero values that are not meaningful routine vital-sign measurements.

**Explanation:** Treating these zeros as actual measurements would distort statistics and downstream analyses involving vital signs.

**Counter-Argument:** The cleaning rule is not based on whether a value is zero alone. It is based on whether zero is a meaningful value for that specific clinical measurement.

**Decision:** Convert zero values in the five identified vital-sign variables to `NaN`. All other zero and negative laboratory values remain unchanged.

In [50]:
vital_zero_cols = [
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "map_value",
    "pulse",
    "respiration"
]

# Replace invalid zero values with missing values
for col in vital_zero_cols:
    labs.loc[labs[col] == 0, col] = np.nan

datasets["Labs"] = labs

print(
    "Remaining zeros in vital columns:",
    (labs[vital_zero_cols] == 0).sum().sum()
)

print("\nNulls introduced per column:")
print(labs[vital_zero_cols].isna().sum())

Remaining zeros in vital columns: 0

Nulls introduced per column:
systolic_blood_pressure     3
diastolic_blood_pressure    3
map_value                   3
pulse                       1
respiration                 1
dtype: int64


### Step 4 — Conclusion

The five identified routine vital-sign variables were cleaned by converting their invalid zero values to missing values. A total of 11 zero observations were affected.
Other zero and negative laboratory values were preserved because they may represent valid clinical measurements.
The identified invalid-value issue is now resolved, and the Labs dataset is ready for final validation and export.

In [51]:
### Final Formatting & Structure Review

print("Number of columns:", len(labs.columns))
print("Duplicate column names:", labs.columns.duplicated().sum())

print("\nData types:")
print(labs.dtypes.value_counts())

print("\nNon-numeric columns:")
print(labs.select_dtypes(exclude=np.number).columns.tolist())

print("\nPotential temporary/investigation columns:")
print([
    col for col in labs.columns
    if any(term in col.lower() for term in ["flag", "check", "temp", "test", "validation"])
])

print("\nColumn name sample:")
print(labs.columns.tolist())

Number of columns: 107
Duplicate column names: 0

Data types:
float64    104
int64        2
bool         1
Name: count, dtype: int64

Non-numeric columns:
['blood_gas_panel_observed']

Potential temporary/investigation columns:
['body_temperature', 'body_temperature_blood_gas']

Column name sample:
['inpatient_number', 'body_temperature', 'pulse', 'respiration', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'map_value', 'fio2', 'creatinine_enzymatic_method', 'urea', 'uric_acid', 'glomerular_filtration_rate', 'cystatin', 'white_blood_cell', 'monocyte_ratio', 'monocyte_count', 'red_blood_cell', 'coefficient_of_variation_of_red_blood_cell_distribution_width', 'standard_deviation_of_red_blood_cell_distribution_width', 'mean_corpuscular_volume', 'hematocrit', 'lymphocyte_count', 'mean_hemoglobin_volume', 'mean_hemoglobin_concentration', 'mean_platelet_volume', 'basophil_ratio', 'basophil_count', 'eosinophil_ratio', 'eosinophil_count', 'hemoglobin', 'platelet', 'platelet_distributio

### Labs — Step 5: Decimal Precision for Derived Values

**Supporting Argument:** `map_value` and `creatine_kinase_isoenzyme_to_creatine_kinase` are both computed ratios of other Labs columns, not direct instrument readings — confirmed for `map_value` by reproducing its formula exactly from `systolic_blood_pressure`/`diastolic_blood_pressure`. Both carry 13–16 decimal places from floating-point division, unlike every other Labs column (2–3 decimals, genuine measurement precision).

**Explanation:** This is the same false-precision issue we already corrected for BMI — rounding doesn't change the underlying measurement, it just removes division artifacts that overstate precision the source data never had.

**Counter-Argument:** We're not rounding every float column, only the two proven to be derived rather than measured — applying a blanket round would incorrectly truncate genuine instrument precision elsewhere.

**Decision:** Round `map_value` and `creatine_kinase_isoenzyme_to_creatine_kinase` to 2 decimals. No other columns are touched.

In [52]:
labs["map_value"] = labs["map_value"].round(2)
labs["creatine_kinase_isoenzyme_to_creatine_kinase"] = labs["creatine_kinase_isoenzyme_to_creatine_kinase"].round(3)

datasets["Labs"] = labs
print(labs[["map_value", "creatine_kinase_isoenzyme_to_creatine_kinase"]].head())

   map_value  creatine_kinase_isoenzyme_to_creatine_kinase
0      76.67                                          0.22
1      96.67                                          0.16
2      78.67                                          0.38
3      86.00                                           NaN
4      86.00                                          0.11


### Labs — Step 5: Final Validation & Export

**Why this step:** Confirm the intended outcome of the Labs cleaning decisions in one consolidated check, then export the cleaned dataset.

Per the lean cleaning approach, this is the only final validation pass. No additional profiling is performed because the relevant data-quality issues were already addressed in Steps 1–4.

In [53]:
print("Shape:", labs.shape)
print("Unique patients:", labs["inpatient_number"].nunique())
print("Duplicate rows:", labs.duplicated().sum())
print("Duplicate patient IDs:", labs["inpatient_number"].duplicated().sum())
print("cholinesterase present:", "cholinesterase" in labs.columns)
print("blood_gas_panel_observed present:", "blood_gas_panel_observed" in labs.columns)
print("Remaining vital-sign zeros:", (labs[["systolic_blood_pressure","diastolic_blood_pressure",
      "map_value","pulse","respiration"]] == 0).sum().sum())

Shape: (2008, 107)
Unique patients: 2008
Duplicate rows: 0
Duplicate patient IDs: 0
cholinesterase present: False
blood_gas_panel_observed present: True
Remaining vital-sign zeros: 0


In [54]:
labs.to_csv(CLEANED_PATH / "labs_cleaned.csv", index=False)
print("\nSaved:", CLEANED_PATH / "labs_cleaned.csv")

labs_check = pd.read_csv(CLEANED_PATH / "labs_cleaned.csv")
print("Reload check — shape:", labs_check.shape, "| columns match:", list(labs_check.columns) == list(labs.columns))


Saved: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\labs_cleaned.csv
Reload check — shape: (2008, 107) | columns match: True


### Lab Cleaning Validation

The cleaned laboratory data was validated for structural integrity, patient duplication, and remaining invalid vital-sign zero values. No duplicate records or duplicate patient IDs were found, and no remaining vital-sign zeros were identified.

The `blood_gas_panel_observed` field was retained as a valid non-numeric field. No further lab-specific cleaning was required based on the available data and data dictionary.


## CARDIAC COMPLICATIONS — DATA CLEANING
### Cardiac Complications - Step 1: Understand Dataset Structure and Grain

Before cleaning, we need to understand the structure and grain of the dataset so that cleaning decisions do not alter the intended patient-level records.
We will check the dataset dimensions, columns, patient uniqueness, and duplicate records.
We will compare the total number of records with unique `inpatient_number` values and check for duplicate rows.
Confirm that the dataset represents one cardiac-complications record per patient and identify any structural duplication before further cleaning.


In [196]:
# Step 1: Structure and Grain Check

print("Shape:", cardiac.shape)
print("Unique patients:", cardiac["inpatient_number"].nunique())
print("Duplicate rows:", cardiac.duplicated().sum())
print("Duplicate patient IDs:", cardiac["inpatient_number"].duplicated().sum())


Shape: (2008, 14)
Unique patients: 2008
Duplicate rows: 0
Duplicate patient IDs: 0


#### Finding and Decision

The dataset contains 2,008 records for 2,008 unique patients, with no duplicate rows or duplicate patient IDs.
No records are removed at the structural level.
Preserve the complete patient-level cardiac dataset for subsequent cleaning and analysis.

### Cardiac Complications - Step 2: Assess Missing Values

Missing values must be understood before deciding whether to remove, retain, or impute them. High missingness does not automatically indicate poor-quality data.
We will identify the number and percentage of missing values in each variable.
Calculate missing counts and percentages for all columns and review the pattern of missingness.
Distinguish between structural missingness and values that require further investigation, while avoiding unnecessary imputation.

In [197]:
# Step 2: Missing Value Assessment

cardiac_missing = (
    cardiac.isna()
    .sum()
    .to_frame("missing_count")
)

cardiac_missing["missing_percent"] = (
    cardiac_missing["missing_count"] / len(cardiac) * 100
).round(2)

cardiac_missing = cardiac_missing.sort_values(
    "missing_percent",
    ascending=False
)

cardiac_missing

,missing_count,missing_percent
tricuspid_valve_return_pressure,1826,90.94
ea,1615,80.43
mitral_valve_ams,1458,72.61
lvef,1373,68.38
tricuspid_valve_return_velocity,1218,60.66
mitral_valve_ems,1028,51.20
left_ventricular_end_diastolic_diameter_lv,699,34.81
inpatient_number,0,0.00
nyha_cardiac_function_classification,0,0.00
killip_grade,0,0.00


**Finding:** Echocardiographic variables (`tricuspid_valve_return_pressure` 90.9%, `ea` 
80.4%, `mitral_valve_ams` 72.6%, `lvef` 68.4%, `tricuspid_valve_return_velocity` 60.7%, 
`mitral_valve_ems` 51.2%, `left_ventricular_end_diastolic_diameter_lv` 34.7%) have 
substantial missingness — expected, since echo is not performed on every admission.

**Decision:** Missing clinical measurements are retained as missing. We will not 
mean/median/mode-impute these, since that would fabricate clinical readings that were 
never taken.

### Cardiac Complications - Step 3: Validate Clinical Classification Variables

Clinical classification variables should contain valid categories before they are used in descriptive or predictive analysis.
We will inspect the observed values of NYHA classification, Killip grade, myocardial infarction, congestive heart failure, peripheral vascular disease, and heart-failure type.
Review the unique non-missing values in each classification variable and compare them with the definitions provided in the data dictionary.
Confirm that the observed categories are valid and avoid changing clinically meaningful values without supporting evidence.

In [198]:
# Step 3: Clinical Classification Validation

classification_columns = [
    "nyha_cardiac_function_classification",
    "killip_grade",
    "myocardial_infarction",
    "congestive_heart_failure",
    "peripheral_vascular_disease",
    "type_of_heart_failure"
]

for column in classification_columns:
    print(f"\n{column}:")
    print(sorted(cardiac[column].dropna().unique()))


nyha_cardiac_function_classification:
[np.int64(2), np.int64(3), np.int64(4)]

killip_grade:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

myocardial_infarction:
[np.int64(0), np.int64(1)]

congestive_heart_failure:
[np.int64(0), np.int64(1)]

peripheral_vascular_disease:
[np.int64(0), np.int64(1)]

type_of_heart_failure:
['Both', 'Left', 'Right']


#### Finding and Decision

The observed classification values are consistent with the expected clinical categories in the dataset.

The data dictionary defines Killip grade as a four-level clinical classification from Class 1 through Class 4. Therefore, observed Killip values are retained as recorded. 
No clinical classification values are overwritten or reassigned.
Preserve the original clinical classifications for downstream analysis.

### Cardiac Complications - Step 4: Investigate Potential Measurement Errors — Mitral Valve E/A Waves

Clinical measurements should be checked for values inconsistent with the 
expected physiological scale before being used in analysis.

`mitral_valve_ems` (E-wave) and `mitral_valve_ams` (A-wave), both defined by 
the data dictionary as velocities in m/s.

 Identify unusually large values and compare against the overall distribution 
to determine whether they're genuine readings or a systematic data-entry issue.

In [199]:
print("Mitral E-wave values >= 10:", (cardiac["mitral_valve_ems"] >= 10).sum())
print("Mitral A-wave values >= 10:", (cardiac["mitral_valve_ams"] >= 10).sum())
print("Maximum E-wave:", cardiac["mitral_valve_ems"].max())
print("Maximum A-wave:", cardiac["mitral_valve_ams"].max())

display(cardiac.loc[
    (cardiac["mitral_valve_ems"] >= 10) | (cardiac["mitral_valve_ams"] >= 10),
    ["inpatient_number", "mitral_valve_ems", "mitral_valve_ams"]
])

Mitral E-wave values >= 10: 0
Mitral A-wave values >= 10: 0
Maximum E-wave: 4.09
Maximum A-wave: 4.08


,inpatient_number,mitral_valve_ems,mitral_valve_ams


**Finding:** 11 E-wave values and 7 A-wave values are ≥10, reaching as high as 409.0 
and 408.0, against a main distribution of roughly 0.6–1.3. These anomalous values show 
a consistent decimal-entry pattern — 409 → 4.09, 403 → 4.03, 44 → 0.44, 99 → 0.99 — all 
of which land back inside the plausible clinical range once divided by 100.

**Decision:** Divide only the flagged values (≥10) by 100, in these two columns only. 
No other values are touched.

In [200]:
for column in ["mitral_valve_ems", "mitral_valve_ams"]:
    mask = cardiac[column] >= 10
    cardiac.loc[mask, column] = cardiac.loc[mask, column] / 100

print("Mitral E-wave values >= 10 after correction:", (cardiac["mitral_valve_ems"] >= 10).sum())
print("Mitral A-wave values >= 10 after correction:", (cardiac["mitral_valve_ams"] >= 10).sum())
print("New maximum E-wave:", cardiac["mitral_valve_ems"].max())
print("New maximum A-wave:", cardiac["mitral_valve_ams"].max())

Mitral E-wave values >= 10 after correction: 0
Mitral A-wave values >= 10 after correction: 0
New maximum E-wave: 4.09
New maximum A-wave: 4.08


### Cardiac Complications - Step 4b: Investigate Potential Measurement Errors — LV End-Diastolic Diameter

Same rationale as Step 4 — check for values outside a physiologically 
plausible range.

`left_ventricular_end_diastolic_diameter_lv` (LVEDD). The data dictionary 
defines LVEDD as a left ventricular end-diastolic diameter with a reference range of 
3.5–5.6 cm. The observed dataset values are numerically represented around 45–60, 
indicating that the dataset appears to use a millimeter-scale representation for this 
variable.

Identify values below 10 and inspect the corresponding patient records.

In [201]:
print(cardiac["left_ventricular_end_diastolic_diameter_lv"].describe())
print("\nValues < 10:", (cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10).sum())

display(cardiac.loc[
    cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10,
    ["inpatient_number", "left_ventricular_end_diastolic_diameter_lv"]
])

count    1309.000000
mean       53.190222
std        10.738113
min        22.000000
25%        45.000000
50%        53.000000
75%        60.000000
max        88.000000
Name: left_ventricular_end_diastolic_diameter_lv, dtype: float64

Values < 10: 0


,inpatient_number,left_ventricular_end_diastolic_diameter_lv


**Finding:** Two patients have LVEDD values below 10 — patient 805096: 0.93, patient 
743698: 0.30 — far below the scale of the remaining observations and inconsistent with 
a physiologically plausible ventricular diameter. Unlike the mitral E/A measurements, 
these two values show no consistent scaling pattern that can be confidently corrected.

**Decision:** Treat these two values as missing (`NaN`) rather than guess a correction, 
consistent with the "don't invent clinical values" approach used elsewhere in this table.

In [202]:
mask = cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10
cardiac.loc[mask, "left_ventricular_end_diastolic_diameter_lv"] = pd.NA

print("Remaining values < 10:", (cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10).sum())
print("Missing LVEDD count now:", cardiac["left_ventricular_end_diastolic_diameter_lv"].isna().sum())

Remaining values < 10: 0
Missing LVEDD count now: 699


### Cardiac Complications - Step 4c: Validate Consistency of `ea` Against Corrected E/A Values

`ea` (E/A ratio) already exists in the source data. Since Step 4 corrected a 
decimal-entry error in the underlying E and A wave values, it's worth checking whether 
`ea` was affected by the same issue before deciding whether it needs any change.

In [203]:
both_present = cardiac.dropna(subset=["ea", "mitral_valve_ems", "mitral_valve_ams"])
computed_ratio = both_present["mitral_valve_ems"] / both_present["mitral_valve_ams"]
diff = (computed_ratio - both_present["ea"]).abs()

print("Rows with ea, ems, and ams all present:", len(both_present))
print("Rows where computed ratio matches ea within 0.05:", (diff < 0.05).sum())

Rows with ea, ems, and ams all present: 392
Rows where computed ratio matches ea within 0.05: 381


**Finding:** 381 of 392 rows (97%) with all three values present match the E/A ratio 
computed from the corrected `mitral_valve_ems`/`mitral_valve_ams` to within 0.05.

**Decision:** `ea` is internally consistent with the corrected mitral values and was 
not affected by the same decimal-entry issue. No recalculation applied.

### Cardiac Complications - Step 5: Final Validation

Confirm the dataset structure remained intact and no unintended changes were 
introduced.

In [204]:
print("Final shape:", cardiac.shape)
print("Unique patients:", cardiac["inpatient_number"].nunique())
print("Duplicate rows:", cardiac.duplicated().sum())
print("Duplicate patient IDs:", cardiac["inpatient_number"].duplicated().sum())
print("\nFinal columns:")
print(cardiac.columns.tolist())

Final shape: (2008, 14)
Unique patients: 2008
Duplicate rows: 0
Duplicate patient IDs: 0

Final columns:
['inpatient_number', 'nyha_cardiac_function_classification', 'killip_grade', 'myocardial_infarction', 'congestive_heart_failure', 'peripheral_vascular_disease', 'type_of_heart_failure', 'lvef', 'left_ventricular_end_diastolic_diameter_lv', 'mitral_valve_ems', 'mitral_valve_ams', 'ea', 'tricuspid_valve_return_velocity', 'tricuspid_valve_return_pressure']


### Cardiac Complications - Step 6: Export Cleaned Cardiac Dataset
The cleaned dataset should be saved separately so that the original raw data remains unchanged and the cleaned version can be used consistently in downstream analysis.

In [205]:
# Step 6: Export cleaned cardiac dataset

cardiac.to_csv(CLEANED_PATH / "cardiac_complications_cleaned.csv", index=False)
print("Cleaned cardiac dataset saved to:", CLEANED_PATH / "cardiac_complications_cleaned.csv")

Cleaned cardiac dataset saved to: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\cardiac_complications_cleaned.csv


In [206]:
cardiac_check = pd.read_csv(
    CLEANED_PATH / "cardiac_complications_cleaned.csv"
)

print("Saved shape:", cardiac_check.shape)
print("Saved duplicate rows:", cardiac_check.duplicated().sum())
print("Saved columns:", cardiac_check.columns.tolist())

Saved shape: (2008, 14)
Saved duplicate rows: 0
Saved columns: ['inpatient_number', 'nyha_cardiac_function_classification', 'killip_grade', 'myocardial_infarction', 'congestive_heart_failure', 'peripheral_vascular_disease', 'type_of_heart_failure', 'lvef', 'left_ventricular_end_diastolic_diameter_lv', 'mitral_valve_ems', 'mitral_valve_ams', 'ea', 'tricuspid_valve_return_velocity', 'tricuspid_valve_return_pressure']


### Cardiac Complications Cleaning — Summary

The cardiac dataset was reviewed for structural integrity, missing values, clinical 
classification consistency, and potential measurement errors.

#### Key findings
- 2,008 patient records retained; no duplicate rows or patient IDs.
- Missing echocardiographic measurements (35–91% missing depending on variable) were 
  retained rather than artificially imputed.
- Clinical classification variables matched the data dictionary; no unsupported 
  recoding (e.g. a Killip-grade override) was applied.
- 18 mitral valve E/A-wave values showing a decimal-entry pattern (values in the 
  hundreds) were corrected by dividing by 100.
- 2 physiologically impossible LV end-diastolic diameter values were set to missing.
- The existing `ea` (E/A ratio) column was validated as consistent with the corrected 
  mitral values and left unchanged.

#### Outcome
The cleaned cardiac dataset preserves original patient-level clinical information while 
correcting only issues supported by evidence, and is ready for descriptive, 
prescriptive, and predictive analysis.

## Patient History Dataset

The Patient History dataset contains patient-level medical history and comorbidity information. Since the common dataset setup and imports were already completed earlier in the cleaning workflow, we use the existing project path and loading approach here.

The focus of this section is to validate Patient History-specific data quality and investigate any issues identified during profiling.


### Patient History — Step 1: Structure, Data Types & Duplicate Check

**Why this step:** Same opening move as every other table — confirm shape, dtypes, and both duplication risks before looking at values.

In [80]:
patient_history = pd.read_csv(DATA_PATH / "patienthistory.csv")

print("Shape:", patient_history.shape)
print("\nData types:")
print(patient_history.dtypes)
display(patient_history.head())

print("\nFull-row duplicates:", patient_history.duplicated().sum())
print("Duplicate patient IDs:", patient_history["inpatient_number"].duplicated().sum())

master_ids = set(hospitalization["inpatient_number"])
diff = set(patient_history["inpatient_number"]) ^ master_ids
print("Patient ID mismatch vs. master population:", diff if diff else "None")

datasets["Patient History"] = patient_history

Shape: (2008, 17)

Data types:
inpatient_number                               int64
cerebrovascular_disease                        int64
dementia                                       int64
chronic_obstructive_pulmonary_disease          int64
connective_tissue_disease                      int64
peptic_ulcer_disease                         float64
diabetes                                       int64
moderate_to_severe_chronic_kidney_disease    float64
hemiplegia                                     int64
leukemia                                       int64
malignant_lymphoma                             int64
solid_tumor                                    int64
liver_disease                                float64
aids                                           int64
cci_score                                    float64
type_ii_respiratory_failure                   object
acute_renal_failure                            int64
dtype: object


,inpatient_number,cerebrovascular_disease,dementia,chronic_obstructive_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,diabetes,moderate_to_severe_chronic_kidney_disease,hemiplegia,leukemia,malignant_lymphoma,solid_tumor,liver_disease,aids,cci_score,type_ii_respiratory_failure,acute_renal_failure
0,857781,0,0,1,0,0.0,1,0.0,0,0,0,0,0.0,0,2.0,nontypeii,0
1,743087,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
2,866418,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
3,775928,0,0,1,0,0.0,0,1.0,0,0,0,0,0.0,0,2.0,nontypeii,0
4,810128,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0



Full-row duplicates: 0
Duplicate patient IDs: 0
Patient ID mismatch vs. master population: None


### Patient History — Step 2: Missingness & Value Profile

**Why this step:** Establish where missing values sit and what every categorical/binary column actually contains, before deciding anything.

In [81]:
missing_report = pd.DataFrame({
    "Missing Count": patient_history.isna().sum(),
    "Missing %": (patient_history.isna().mean() * 100).round(2),
    "Data Type": patient_history.dtypes.astype(str)
})
missing_report = missing_report[missing_report["Missing Count"] > 0].sort_values("Missing %", ascending=False)
display(missing_report)

for col in ["cerebrovascular_disease","dementia","chronic_obstructive_pulmonary_disease","connective_tissue_disease",
            "peptic_ulcer_disease","diabetes","moderate_to_severe_chronic_kidney_disease","hemiplegia","leukemia",
            "malignant_lymphoma","solid_tumor","liver_disease","aids","acute_renal_failure","cci_score",
            "type_ii_respiratory_failure"]:
    print(f"\n{col}:")
    print(patient_history[col].value_counts(dropna=False))

,Missing Count,Missing %,Data Type
cci_score,5,0.25,float64
peptic_ulcer_disease,2,0.10,float64
moderate_to_severe_chronic_kidney_disease,2,0.10,float64
liver_disease,1,0.05,float64



cerebrovascular_disease:
cerebrovascular_disease
0    1858
1     150
Name: count, dtype: int64

dementia:
dementia
0    1893
1     115
Name: count, dtype: int64

chronic_obstructive_pulmonary_disease:
chronic_obstructive_pulmonary_disease
0    1775
1     233
Name: count, dtype: int64

connective_tissue_disease:
connective_tissue_disease
0    2004
1       4
Name: count, dtype: int64

peptic_ulcer_disease:
peptic_ulcer_disease
0.0    1961
1.0      45
NaN       2
Name: count, dtype: int64

diabetes:
diabetes
0    1542
1     466
Name: count, dtype: int64

moderate_to_severe_chronic_kidney_disease:
moderate_to_severe_chronic_kidney_disease
0.0    1532
1.0     474
NaN       2
Name: count, dtype: int64

hemiplegia:
hemiplegia
0    1996
1      12
Name: count, dtype: int64

leukemia:
leukemia
0    2008
Name: count, dtype: int64

malignant_lymphoma:
malignant_lymphoma
0    2007
1       1
Name: count, dtype: int64

solid_tumor:
solid_tumor
0    1969
1      39
Name: count, dtype: int64

liver_dis

### Patient History — Step 3: Missingness-Based Cleaning Decision

**Supporting Argument:** 5 patients each have exactly one missing comorbidity flag (`peptic_ulcer_disease`: 2, `moderate_to_severe_chronic_kidney_disease`: 2, `liver_disease`: 1) and are also missing `cci_score` — a perfect 1:1 correspondence, confirmed directly against the data: no patient is missing `cci_score` without a missing comorbidity input, and none is missing a comorbidity input without also missing `cci_score`. This indicates `cci_score` likely could not be computed when one of its component flags was blank.

**Explanation:** We can't confirm whether a blank comorbidity flag means "confirmed absent" or "never assessed." Filling with 0 would silently convert 5 unknowns into false negatives for a clinical history variable.

**Counter-Argument:** At 0.25% of the cohort, someone could argue for filling with 0 as a low-risk default. We reject that — the entire reason for not blanket-imputing binary medical-history variables is to avoid exactly this kind of unverified

for col in [
    "peptic_ulcer_disease",
    "moderate_to_severe_chronic_kidney_disease",
    "liver_disease"
]:
    patient_history[col] = patient_history[col].astype("Int64")

print(
    patient_history[
        ["peptic_ulcer_disease",
         "moderate_to_severe_chronic_kidney_disease",
         "liver_disease"]
    ].dtypes
)

### Step 4: Investigate Potential Measurement Errors — Mitral Valve E/A Waves
Clinical measurements should be checked for values outside a physiologically 
plausible range before being used in analysis.

Mitral valve E-wave (`mitral_valve_ems`) and A-wave (`mitral_valve_ams`) 
velocities, both measured in m/s. The data dictionary defines both as velocities, so 
values in the hundreds are not physiologically possible.


In [174]:
print("Mitral E-wave values >= 10:", (cardiac["mitral_valve_ems"] >= 10).sum())
print("Mitral A-wave values >= 10:", (cardiac["mitral_valve_ams"] >= 10).sum())
print("Maximum E-wave:", cardiac["mitral_valve_ems"].max())
print("Maximum A-wave:", cardiac["mitral_valve_ams"].max())

display(cardiac.loc[
    (cardiac["mitral_valve_ems"] >= 10) | (cardiac["mitral_valve_ams"] >= 10),
    ["inpatient_number", "mitral_valve_ems", "mitral_valve_ams"]
])

Mitral E-wave values >= 10: 11
Mitral A-wave values >= 10: 7
Maximum E-wave: 409.0
Maximum A-wave: 408.0


,inpatient_number,mitral_valve_ems,mitral_valve_ams
52,791903,0.52,408.00
57,731805,409.00,0.69
79,841390,0.88,401.00
141,803586,401.00,0.89
151,786403,1.16,40.00
158,848849,405.00,1.05
171,780942,407.00,0.97
189,802669,401.00,0.53
191,835944,1.32,40.00
226,808801,0.98,401.00


**Finding:** 11 values in `mitral_valve_ems` and 7 values in `mitral_valve_ams` fall 
between 36 and 409 — roughly 30–400x the rest of the distribution (25th–75th 
percentile: 0.6–1.3). Dividing these specific values by 100 places every one of them 
back inside the plausible clinical range (e.g. 409 → 4.09, 44 → 0.44), which is a 
strong, consistent signature of a systematic decimal-entry error rather than genuine 
extreme readings.

**Decision:** Divide only the flagged values (≥10) by 100, in these two columns only. 
No other values are touched.

In [175]:
for column in ["mitral_valve_ems", "mitral_valve_ams"]:
    mask = cardiac[column] >= 10
    cardiac.loc[mask, column] = cardiac.loc[mask, column] / 100

print("Mitral E-wave values >= 10 after correction:", (cardiac["mitral_valve_ems"] >= 10).sum())
print("Mitral A-wave values >= 10 after correction:", (cardiac["mitral_valve_ams"] >= 10).sum())
print("New max E-wave:", cardiac["mitral_valve_ems"].max())
print("New max A-wave:", cardiac["mitral_valve_ams"].max())

Mitral E-wave values >= 10 after correction: 0
Mitral A-wave values >= 10 after correction: 0
New max E-wave: 4.09
New max A-wave: 4.08


#### Step 4b: Investigate Potential Measurement Errors — LV End-Diastolic Diameter

Same rationale as Step 4 — check for values outside a physiologically 
plausible range.
`left_ventricular_end_diastolic_diameter_lv`, measured in mm per the data 
dictionary. Normal-to-severely-dilated range is roughly 35–90mm; the observed 
distribution here (median 53, IQR 45–60, max 88) is consistent with that.

In [176]:
print(cardiac["left_ventricular_end_diastolic_diameter_lv"].describe())
print("\nValues < 10:", (cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10).sum())

display(cardiac.loc[
    cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10,
    ["inpatient_number", "left_ventricular_end_diastolic_diameter_lv"]
])

count    1311.000000
mean       53.110015
std        10.924505
min         0.300000
25%        45.000000
50%        53.000000
75%        60.000000
max        88.000000
Name: left_ventricular_end_diastolic_diameter_lv, dtype: float64

Values < 10: 2


,inpatient_number,left_ventricular_end_diastolic_diameter_lv
172,805096,0.93
1133,743698,0.30


**Finding:** Two patients (805096, 743698) have values of 0.93 and 0.30 — far below any 
physiologically possible LV diastolic diameter, and inconsistent with the rest of the 
column (min of the remaining values is well above 30). Unlike the mitral valve case, 
there's no clean scaling factor that brings these into range, so this looks like a 
data-entry/measurement error rather than a decimal-shift.

**Decision:** Set these two values to missing (`NaN`) rather than guess a correction. 
This preserves the "don't invent clinical values" principle used for the rest of this 
table's missingness.

In [177]:
mask = cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10
cardiac.loc[mask, "left_ventricular_end_diastolic_diameter_lv"] = pd.NA

print("Remaining values < 10:", (cardiac["left_ventricular_end_diastolic_diameter_lv"] < 10).sum())
print("Missing count now:", cardiac["left_ventricular_end_diastolic_diameter_lv"].isna().sum())

Remaining values < 10: 0
Missing count now: 699


### Patient History — Step 5: Final Validation & Export

In [178]:
# Remove analysis-only derived column from the cleaned dataset
if "comorbidity_count" in patient_history.columns:
    patient_history = patient_history.drop(columns=["comorbidity_count"])

print("Final shape:", patient_history.shape)

Final shape: (2008, 17)


In [179]:
print("Shape:", patient_history.shape)
print("Unique patients:", patient_history["inpatient_number"].nunique())
print("Duplicate rows:", patient_history.duplicated().sum())
print("Duplicate patient IDs:", patient_history["inpatient_number"].duplicated().sum())

print("\nMissing values remaining:")
print(patient_history.isna().sum()[patient_history.isna().sum() > 0])

print("\nData types:")
print(patient_history.dtypes)

datasets["Patient History"] = patient_history

patient_history_cleaned_path = CLEANED_PATH / "patienthistory_cleaned.csv"
patient_history.to_csv(patient_history_cleaned_path, index=False)

print("\nSaved:", patient_history_cleaned_path)

ph_check = pd.read_csv(patient_history_cleaned_path)
print("\nReload check — shape:", ph_check.shape)

Shape: (2008, 17)
Unique patients: 2008
Duplicate rows: 0
Duplicate patient IDs: 0

Missing values remaining:
peptic_ulcer_disease                         2
moderate_to_severe_chronic_kidney_disease    2
liver_disease                                1
cci_score                                    5
dtype: int64

Data types:
inpatient_number                               int64
cerebrovascular_disease                        int64
dementia                                       int64
chronic_obstructive_pulmonary_disease          int64
connective_tissue_disease                      int64
peptic_ulcer_disease                         float64
diabetes                                       int64
moderate_to_severe_chronic_kidney_disease    float64
hemiplegia                                     int64
leukemia                                       int64
malignant_lymphoma                             int64
solid_tumor                                    int64
liver_disease                           

### Patient History — Step 5: Final Validation & Export

**Why this step:** Confirm that the cleaned dataset was saved correctly and that no unintended duplicates or structural changes were introduced during cleaning.

The final dataset is validated for shape, duplicate records, duplicate patient IDs, and preserved missing values before being used for downstream analysis.


In [85]:
patient_history_cleaned_path = CLEANED_PATH / "patienthistory_cleaned.csv"
patient_history.to_csv(patient_history_cleaned_path, index=False)

print("Saved:", patient_history_cleaned_path)

Saved: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patienthistory_cleaned.csv


In [86]:
patient_history_check = pd.read_csv(
    CLEANED_PATH / "patienthistory_cleaned.csv"
)

print("Saved shape:", patient_history_check.shape)
print("Duplicate rows:", patient_history_check.duplicated().sum())
print("Duplicate patient IDs:", patient_history_check["inpatient_number"].duplicated().sum())
print("Missing values:", patient_history_check.isna().sum().sum())
print("Saved columns:", patient_history_check.columns.tolist())

Saved shape: (2008, 17)
Duplicate rows: 0
Duplicate patient IDs: 0
Missing values: 10
Saved columns: ['inpatient_number', 'cerebrovascular_disease', 'dementia', 'chronic_obstructive_pulmonary_disease', 'connective_tissue_disease', 'peptic_ulcer_disease', 'diabetes', 'moderate_to_severe_chronic_kidney_disease', 'hemiplegia', 'leukemia', 'malignant_lymphoma', 'solid_tumor', 'liver_disease', 'aids', 'cci_score', 'type_ii_respiratory_failure', 'acute_renal_failure']


### Patient History — Cleaning Conclusion

The Patient History dataset was cleaned without altering valid clinical information. The dataset contains **2,008 patients and 17 columns**, with no duplicate records or duplicate patient IDs.

Binary comorbidity fields were validated for consistent `0/1` values, while `CCI_score` contained valid scores ranging from 0–6. Five patients had missing CCI scores, each corresponding to a missing comorbidity input. Because the available documentation does not confirm that a missing medical-history value represents absence of disease, these values were preserved as missing rather than replaced with `0`.

No invalid values required correction, and the cleaned dataset was exported for downstream analysis.


### Patient Prescriptions Table


In [128]:
# Restore the real Patient Prescriptions dataset
patient_prescriptions = pd.read_csv(
    DATA_PATH / "patient_precriptions.csv"
)

print("Restored shape:", patient_prescriptions.shape)
print("Columns:", patient_prescriptions.columns.tolist())

Restored shape: (15362, 2)
Columns: ['inpatient_number', 'drug_name']


### Patient Prescriptions — Step 1: Structure & Grain Check

**Why this step:** Unlike the other six tables, Patient Prescriptions is one-to-many 
(one row per prescribed drug, not one row per patient), so it needs its own grain-aware 
checks rather than the standard patient-level duplicate check. 

In [129]:
print("Patient Prescriptions shape:", patient_prescriptions.shape)
print("Unique patients:", patient_prescriptions["inpatient_number"].nunique())
print("Unique drugs:", patient_prescriptions["drug_name"].nunique())

print("\nData types:")
print(patient_prescriptions.dtypes)

print("\nSample records:")
display(patient_prescriptions.head())

print(
    "\nRows per patient (min / median / max):",
    patient_prescriptions.groupby("inpatient_number").size().min(),
    patient_prescriptions.groupby("inpatient_number").size().median(),
    patient_prescriptions.groupby("inpatient_number").size().max(),
)

Patient Prescriptions shape: (15362, 2)
Unique patients: 2007
Unique drugs: 25

Data types:
inpatient_number     int64
drug_name           object
dtype: object

Sample records:


,inpatient_number,drug_name
0,857781,sulfotanshinone sodium injection
1,857781,Furosemide tablet
2,857781,Enoxaparin Sodium injection
3,857781,Meglumine Adenosine Cyclophosphate for injection
4,857781,Furosemide injection



Rows per patient (min / median / max): 1 8.0 16


### Patient Prescriptions — Step 2: Missing Values & Duplicate Check

**Why this step:** Before touching any values, confirm whether there are missing 
entries, exact duplicate rows, or a patient prescribed the same drug more than once 
(which would be a legitimate multi-row case, not necessarily an error, but worth seeing).

In [116]:
print("Missing values:")
print(patient_prescriptions.isna().sum())

print("\nExact duplicate rows:", patient_prescriptions.duplicated().sum())

print(
    "\nDuplicate (patient, drug) pairs:",
    patient_prescriptions.duplicated(subset=["inpatient_number", "drug_name"]).sum()
)

master_ids = set(hospitalization["inpatient_number"])
missing_from_prescriptions = master_ids - set(patient_prescriptions["inpatient_number"])
print("\nPatients with no prescription record:", missing_from_prescriptions)

Missing values:
inpatient_number    0
drug_name           0
dtype: int64

Exact duplicate rows: 0

Duplicate (patient, drug) pairs: 0

Patients with no prescription record: {741377, 823297, 860163, 761860, 741379, 798725, 860169, 802827, 745485, 851982, 733199, 778259, 770068, 823317, 823318, 831510, 745505, 815140, 753702, 790571, 815148, 729142, 729143, 864312, 794681, 815164, 757820, 811076, 741444, 798791, 864327, 741449, 786506, 761934, 827471, 831570, 737365, 770134, 860246, 815191, 770137, 835675, 733275, 802911, 749668, 802917, 831589, 811111, 741476, 839791, 737392, 815219, 737396, 794740, 761974, 843892, 852088, 798835, 831603, 815223, 790648, 770173, 733308, 852097, 823427, 770180, 761989, 802949, 807047, 786568, 823432, 831625, 860301, 811150, 856206, 860305, 827539, 782485, 823447, 852120, 737433, 757913, 741530, 856223, 794784, 798882, 831651, 860325, 790695, 807080, 782504, 835754, 848044, 852141, 807084, 786611, 807093, 737462, 741559, 729272, 827576, 827580, 794825, 79

### Patient Prescriptions — Step 3: Value Inspection

**Why this step:** Check the actual drug name labels for whitespace and casing issues 
before deciding whether any standardization is needed. This is investigation only — 
no values are changed here.

In [130]:
has_whitespace = patient_prescriptions["drug_name"].apply(lambda x: x != x.strip())
print("Rows with leading/trailing whitespace:", has_whitespace.sum())

lower_stripped = patient_prescriptions["drug_name"].str.strip().str.lower()
print("Unique drugs (raw):", patient_prescriptions["drug_name"].nunique())
print("Unique drugs (case/whitespace-insensitive):", lower_stripped.nunique())

print("\nAll drug names, sorted:")
print(sorted(patient_prescriptions["drug_name"].unique()))

print("\nFull drug frequency:")
print(patient_prescriptions["drug_name"].value_counts())

Rows with leading/trailing whitespace: 0
Unique drugs (raw): 25
Unique drugs (case/whitespace-insensitive): 25

All drug names, sorted:
['Aspirin enteric-coated tablet', 'Atorvastatin calcium tablet', 'Benazepril hydrochloride tablet', 'Clopidogrel Hydrogen Sulphate tablet', 'Deslanoside injection', 'Digoxin tablet', 'Dobutamine hydrochloride injection', 'Enoxaparin Sodium injection', 'Furosemide injection', 'Furosemide tablet', 'Heparin Sodium injection', 'Hydrochlorothiazide tablet', 'Isoprenaline Hydrochloride injection', 'Isosorbide Mononitrate Sustained Release tablet', 'Meglumine Adenosine Cyclophosphate for injection', 'Metoprolol Succinate Sustained-release tablet', 'Milrinone injection', 'Nitroglycerin injection', 'Shenfu injection', 'Spironolactone tablet', 'Torasemide tablet', 'Valsartan Dispersible tablet', 'metoprolol tartrate injection', 'sulfotanshinone sodium injection', 'warfarin sodium tablet']

Full drug frequency:
drug_name
Spironolactone tablet                     

### Patient Prescriptions — Step 4: Cleaning Decision

**Supporting Argument:** Step 3 confirms there are no missing values, no duplicate 
rows, no whitespace-padded entries, and — critically — no drug that appears under two 
different spellings or cases. All 25 drug names are already distinct at the case- and 
whitespace-insensitive level, so there is no group-by fracturing to fix. The only 
inconsistency is that 3 of 25 drug names (`metoprolol tartrate injection`, 
`sulfotanshinone sodium injection`, `warfarin sodium tablet`) start lowercase while the 
other 22 are Title Case.

**Explanation:** Standardizing capitalization improves readability and presentation 
consistency without merging or altering any clinical meaning, since it does not change 
which drugs are considered distinct.

**Counter-Argument:** One could argue no change is needed at all, since the 
inconsistency doesn't affect grouping or counts. We standardize anyway because 
presentation-only cleaning has no downside here and the hackathon rubric rewards a 
consistently formatted, presentation-ready output — but we do **not** deduplicate or 
merge drug categories, since none are duplicated.

**Decision:** Apply `.str.strip().str.title()` to `drug_name` for consistent casing 
only. No rows are removed, no drugs are merged, and patient `789308`'s absence from 
this table is preserved as-is (already established during the cross-dataset audit as 
a valid patient with no prescription record, not a data error).

In [131]:
drug_case_fixes = {
    "metoprolol tartrate injection": "Metoprolol Tartrate Injection",
    "sulfotanshinone sodium injection": "Sulfotanshinone Sodium Injection",
    "warfarin sodium tablet": "Warfarin Sodium Tablet"
}

patient_prescriptions["drug_name"] = (
    patient_prescriptions["drug_name"].replace(drug_case_fixes)
)

print("Unique drugs after standardization:", patient_prescriptions["drug_name"].nunique())
print(sorted(patient_prescriptions["drug_name"].unique()))

Unique drugs after standardization: 25
['Aspirin enteric-coated tablet', 'Atorvastatin calcium tablet', 'Benazepril hydrochloride tablet', 'Clopidogrel Hydrogen Sulphate tablet', 'Deslanoside injection', 'Digoxin tablet', 'Dobutamine hydrochloride injection', 'Enoxaparin Sodium injection', 'Furosemide injection', 'Furosemide tablet', 'Heparin Sodium injection', 'Hydrochlorothiazide tablet', 'Isoprenaline Hydrochloride injection', 'Isosorbide Mononitrate Sustained Release tablet', 'Meglumine Adenosine Cyclophosphate for injection', 'Metoprolol Succinate Sustained-release tablet', 'Metoprolol Tartrate Injection', 'Milrinone injection', 'Nitroglycerin injection', 'Shenfu injection', 'Spironolactone tablet', 'Sulfotanshinone Sodium Injection', 'Torasemide tablet', 'Valsartan Dispersible tablet', 'Warfarin Sodium Tablet']


### Patient Prescriptions — Step 5: Post-Cleaning Validation

**Why this step:** Confirm the standardization did not change the number of distinct 
drugs, the row count, or the patient population, since this was intended to be a 
cosmetic-only change.

In [133]:
print("Final shape:", patient_prescriptions.shape)
print("Unique patients:", patient_prescriptions["inpatient_number"].nunique())
print("Unique drugs:", patient_prescriptions["drug_name"].nunique())
print("Exact duplicate rows:", patient_prescriptions.duplicated().sum())
print("Missing values:", patient_prescriptions.isna().sum().sum())

assert patient_prescriptions.shape == (15362, 2)
assert patient_prescriptions["drug_name"].nunique() == 25
assert patient_prescriptions.duplicated().sum() == 0
assert patient_prescriptions.isna().sum().sum() == 0

print("\nAll structural validation checks passed.")

datasets["Patient Prescriptions"] = patient_prescriptions

Final shape: (15362, 2)
Unique patients: 2007
Unique drugs: 25
Exact duplicate rows: 0
Missing values: 0

All structural validation checks passed.


### Patient Prescriptions — Step 6: Export Cleaned Dataset

In [136]:
patient_prescriptions.to_csv(
    CLEANED_PATH / "patient_prescriptions_cleaned.csv",
    index=False
)

print(
    "Saved:",
    CLEANED_PATH / "patient_prescriptions_cleaned.csv"
)

Saved: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patient_prescriptions_cleaned.csv


In [135]:
patient_prescriptions_check = pd.read_csv(
    CLEANED_PATH / "patient_prescriptions_cleaned.csv"
)

print("Saved shape:", patient_prescriptions_check.shape)
print("Saved unique drugs:", patient_prescriptions_check["drug_name"].nunique())
print("Saved columns:", patient_prescriptions_check.columns.tolist())

Saved shape: (15362, 2)
Saved unique drugs: 25
Saved columns: ['inpatient_number', 'drug_name']


### Patient Prescriptions — Cleaning Conclusion

The Patient Prescriptions dataset contains 15,362 prescription records across 2,007 
patients (one-to-many grain — patient `789308` has no prescription record, consistent 
with the earlier cross-dataset audit). No missing values, duplicate rows, duplicate 
patient-drug pairs, or whitespace issues were found. All 25 distinct drugs were already 
uniquely identified before cleaning; the only change applied was standardizing 
capitalization for 3 of 25 drug names to Title Case for consistent presentation. No 
records were removed, merged, or imputed.

The cleaned dataset is exported as `patient_prescriptions_cleaned.csv` and is ready for 
downstream analysis.

## Responsiveness — Cleaning and Validation

This section incorporates the responsiveness notebook into the same workflow as the other tables. The master notebook already loads `responsivenes.csv` into `responsiveness` in Step 3. We work on a copy of **that** DataFrame and save the result to the master's `CLEANED_PATH`. The numeric `inpatient_number` is retained so that this table remains compatible with the other patient tables.

The cells below have been checked independently using the supplied responsiveness extract read as a CSV. Run the **whole master notebook** in the team environment after setting its `DATA_PATH` to the folder containing all seven CSV files and the data dictionary; those other files were not provided with this merge request.

The original six source fields are retained. `gcs` is verified as the sum of the eye, verbal, and movement scores; the total is useful for descriptive analysis even though a model using the three component scores would not also need it as an independent feature. A review flag makes unresolved category differences visible without changing the recorded clinical labels.

### Responsiveness — Step 1: Inspect Source Structure

**Why this step:** Confirm that the master loaded the intended six-column CSV before applying the rules from the standalone notebook. Copy it so the source remains available for comparison. The original master uses numeric patient IDs; we validate that representation below rather than converting IDs to strings.

In [100]:
resp_source = responsiveness.copy(deep=True)
resp = resp_source.copy(deep=True)
resp_expected = [
    'inpatient_number', 'eye_opening', 'verbal_response',
    'movement', 'consciousness', 'gcs'
]
print('Responsiveness source shape:', resp_source.shape)
print('Responsiveness source data types:')
print(resp_source.dtypes.to_string())
print('Responsiveness source columns:', resp_source.columns.tolist())

Responsiveness source shape: (2008, 7)
Responsiveness source data types:
inpatient_number                      int64
eye_opening                           int64
verbal_response                       int64
movement                              int64
consciousness                string[python]
gcs                                   int64
consciousness_review_flag              bool
Responsiveness source columns: ['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs', 'consciousness_review_flag']


### Responsiveness — Step 2: Standardize Headers and Validate Patient IDs

**Why this step:** Consistent column names keep the master notebook readable. `inpatient_number` must remain a valid integer so it can be compared or joined with `hospitalization` and the other tables. An invalid or missing ID requires source review; it cannot be guessed.

In [102]:
print("Current responsiveness columns:")
print(resp.columns.tolist())

print("\nExpected responsiveness columns:")
print(resp_expected)

print("\nColumns in resp but not expected:")
print([col for col in resp.columns if col not in resp_expected])

print("\nColumns expected but not in resp:")
print([col for col in resp_expected if col not in resp.columns])

Current responsiveness columns:
['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs', 'consciousness_review_flag']

Expected responsiveness columns:
['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs']

Columns in resp but not expected:
['consciousness_review_flag']

Columns expected but not in resp:
[]


In [190]:
resp_original_headers = resp.columns.tolist()
resp.columns = (resp.columns.astype(str).str.strip().str.lower()
                .str.replace(r'\s+', '_', regex=True))
resp_header_changes = sum(a != b for a, b in zip(resp_original_headers, resp.columns))
if resp.columns.tolist() != resp_expected:
    raise ValueError(f'Unexpected responsiveness columns: {resp.columns.tolist()}')

resp_numeric_ids = pd.to_numeric(resp['inpatient_number'], errors='coerce')
resp_invalid_ids = resp_numeric_ids.isna() | resp_numeric_ids.mod(1).ne(0)
if resp_invalid_ids.any():
    raise ValueError(f'{int(resp_invalid_ids.sum())} responsiveness patient IDs need review.')
resp['inpatient_number'] = resp_numeric_ids.astype('int64')
print('Header names changed:', resp_header_changes)
print('Patient ID type:', resp['inpatient_number'].dtype)

ValueError: Unexpected responsiveness columns: ['inpatient_number', 'eye_opening', 'verbal_response', 'movement', 'consciousness', 'gcs', 'consciousness_review_flag']

### Responsiveness — Step 3: Check Duplicates and Missing Values

**Why this step:** An exact repeated row can inflate counts. A repeated patient ID may represent a different encounter, so do not delete it automatically. Trim categorical text and count missing cells in every field before deciding whether any imputation is justified. This file has neither duplicates nor missing values; no records are removed or fabricated.

In [105]:
resp_exact_duplicates = int(resp.duplicated().sum())
resp = resp.drop_duplicates().reset_index(drop=True)
resp_duplicate_ids = int(resp['inpatient_number'].duplicated().sum())
if resp_duplicate_ids:
    raise ValueError('Repeated responsiveness patient IDs require source review.')

resp['consciousness'] = resp['consciousness'].astype('string').str.strip()
resp['consciousness'] = resp['consciousness'].replace('', pd.NA)
resp_missing_count = resp.isna().sum()
resp_missing_pct = (resp.isna().mean() * 100).round(2)
resp_missing_report = pd.DataFrame({
    'missing_count': resp_missing_count,
    'missing_percent': resp_missing_pct
})
print('Exact duplicate rows removed:', resp_exact_duplicates)
print('Repeated patient IDs:', resp_duplicate_ids)
print(resp_missing_report.to_string())
if resp_missing_count.any():
    raise ValueError('Missing responsiveness values require field-specific review.')

Exact duplicate rows removed: 0
Repeated patient IDs: 0
                           missing_count  missing_percent
inpatient_number                       0              0.0
eye_opening                            0              0.0
verbal_response                        0              0.0
movement                               0              0.0
consciousness                          0              0.0
gcs                                    0              0.0
consciousness_review_flag              0              0.0


### Responsiveness — Step 4: Normalize Recorded Category Labels

**Why this step:** Spaces or casing can split one category across multiple spellings. Standardize only the four labels documented in this extract. Unknown labels cause a clear error, and no patient's `consciousness` category is inferred from GCS. The supplied file needs no category edits.

In [106]:
resp_label_map = {
    'clear': 'Clear',
    'responsivetosound': 'ResponsiveToSound',
    'responsivetopain': 'ResponsiveToPain',
    'nonresponsive': 'Nonresponsive'
}
resp_labels_before = resp['consciousness'].copy()
resp_label_keys = (resp['consciousness']
                   .str.replace(r'\s+', '', regex=True)
                   .str.casefold())
resp_new_labels = resp_label_keys.map(resp_label_map)
if resp_new_labels.isna().any():
    raise ValueError('Unrecognized consciousness label in responsiveness.')
resp['consciousness'] = resp_new_labels.astype('string')
resp_label_changes = int((resp_labels_before != resp['consciousness']).sum())
print('Category formatting changes:', resp_label_changes)
print(resp['consciousness'].value_counts().to_string())

Category formatting changes: 0
consciousness
Clear                1974
ResponsiveToSound      19
Nonresponsive          11
ResponsiveToPain        4


### Responsiveness — Step 5: Validate Score Ranges and the GCS Total

**Why this step:** The Glasgow Coma Scale components must be integers within their permitted ranges: eye opening 1–4, verbal response 1–5, and movement 1–6. The GCS total must be 3–15 and equal the sum of its three components. Reject invalid entries for source review rather than silently changing a clinical score. The supplied file passes these checks.

In [107]:
resp_score_ranges = {
    'eye_opening': (1, 4), 'verbal_response': (1, 5),
    'movement': (1, 6), 'gcs': (3, 15)
}
for resp_column, (resp_low, resp_high) in resp_score_ranges.items():
    resp_numeric = pd.to_numeric(resp[resp_column], errors='coerce')
    resp_invalid = (resp_numeric.isna() | resp_numeric.mod(1).ne(0)
                    | ~resp_numeric.between(resp_low, resp_high))
    print(f'{resp_column}: invalid score count = {int(resp_invalid.sum())}')
    if resp_invalid.any():
        raise ValueError(f'Invalid {resp_column} values require source review.')
    resp[resp_column] = resp_numeric.astype('int64')

resp_computed_gcs = resp['eye_opening'] + resp['verbal_response'] + resp['movement']
resp_gcs_mismatches = int(resp_computed_gcs.ne(resp['gcs']).sum())
print('GCS total mismatches:', resp_gcs_mismatches)
if resp_gcs_mismatches:
    raise ValueError('Recorded GCS differs from its components.')

eye_opening: invalid score count = 0
verbal_response: invalid score count = 0
movement: invalid score count = 0
gcs: invalid score count = 0
GCS total mismatches: 0


### Responsiveness — Step 6: Flag Category Patterns for Review

**Why this step:** Identical GCS component patterns sometimes appear with different `consciousness` labels. The three scores alone do not establish whether a label is wrong; other clinical context may matter. Preserve each recorded label and mark all affected rows with `consciousness_review_flag`. On the supplied data, two patterns affect 29 rows.

In [108]:
resp_components = ['eye_opening', 'verbal_response', 'movement']
resp_distinct_labels = resp.groupby(resp_components)['consciousness'].transform('nunique')
resp['consciousness_review_flag'] = resp_distinct_labels.gt(1)

resp_review_groups = (resp.loc[resp['consciousness_review_flag']]
                      .groupby(resp_components)
                      .agg(rows=('gcs', 'size'), labels=('consciousness', 'nunique'))
                      .reset_index())
print('Score combinations needing review:', len(resp_review_groups))
print('Rows flagged:', int(resp['consciousness_review_flag'].sum()))
print(resp_review_groups.to_string(index=False))

Score combinations needing review: 2
Rows flagged: 29
 eye_opening  verbal_response  movement  rows  labels
           1                1         1    13       4
           3                3         5    16       2


### Responsiveness — Step 7: Check Patient Coverage Against the Master Table

**Why this step:** The master notebook connects its tables through `inpatient_number`, with `hospitalization` as the patient reference. Compare ID sets using the same numeric representation. Show any differences so the team can investigate them; the code does not fabricate or drop records to make the sets match.

In [109]:
resp_reference_ids = set(pd.to_numeric(hospitalization['inpatient_number'], errors='raise'))
resp_clean_ids = set(resp['inpatient_number'])
resp_only_ids = resp_clean_ids - resp_reference_ids
resp_missing_reference_ids = resp_reference_ids - resp_clean_ids
print('Patients in Responsiveness only:', len(resp_only_ids))
print('Patients missing from Responsiveness:', len(resp_missing_reference_ids))

Patients in Responsiveness only: 0
Patients missing from Responsiveness: 0


### Responsiveness — Step 8: Final Validation and Export

**Why this step:** Keep the source data intact, confirm that the cleaned table still has one row per patient, and export alongside the master's other cleaned CSV files. The six source fields and one review flag form a seven-column table. Update `datasets["Responsiveness"]` so later master cells refer to the cleaned version. Finally, reload the CSV to verify its dimensions and IDs.

In [112]:
assert len(resp) == len(resp_source) - resp_exact_duplicates
assert resp['inpatient_number'].is_unique
assert not resp[resp_expected].isna().any().any()
assert resp.columns.tolist() == resp_expected + ['consciousness_review_flag']
assert resp_gcs_mismatches == 0

responsiveness = resp.copy(deep=True)
datasets['Responsiveness'] = responsiveness
responsiveness_cleaned_path = CLEANED_PATH / 'responsivenes_cleaned.csv'
responsiveness.to_csv(responsiveness_cleaned_path, index=False)
resp_saved = pd.read_csv(responsiveness_cleaned_path)
assert resp_saved.shape == responsiveness.shape
assert resp_saved.columns.tolist() == responsiveness.columns.tolist()
assert resp_saved['inpatient_number'].equals(responsiveness['inpatient_number'])
assert int(resp_saved['consciousness_review_flag'].sum()) == int(
    responsiveness['consciousness_review_flag'].sum()
)

print('Cleaned Responsiveness shape:', responsiveness.shape)
print('Original rows retained:', len(responsiveness))
print('Rows flagged for label review:', int(responsiveness['consciousness_review_flag'].sum()))
print('Export verified:', responsiveness_cleaned_path)

Cleaned Responsiveness shape: (2008, 7)
Original rows retained: 2008
Rows flagged for label review: 29
Export verified: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\responsivenes_cleaned.csv


### Responsiveness — Cleaning Conclusion

The supplied responsiveness extract contains 2,008 unique patient records. The field and GCS validation found no missing cells, duplicate IDs, out-of-range scores, or incorrect score totals. The six original fields were retained. A seventh field flags 29 records whose GCS component combinations occur with different recorded consciousness labels; those labels were not guessed or overwritten. The cleaned table is now available as `responsiveness` within this notebook and as `responsivenes_cleaned.csv` under `CLEANED_PATH`.

## Final Cross-Dataset Validation

Before merging or analyzing anything, we're checking that all 7 cleaned files agree 
with each other — same 2,008 patients across the six per-patient tables, no duplicate 
patient IDs, and no one silently dropped or gained during cleaning. Prescriptions is 
checked separately since it's expected to have one patient (789308) missing.

In [207]:
import glob

cleaned_files = sorted(glob.glob(str(CLEANED_PATH / "*_cleaned.csv")))
print("Cleaned files found:")
for f in cleaned_files:
    print(" -", f)

cleaned_dfs = {f: pd.read_csv(f) for f in cleaned_files}

Cleaned files found:
 - C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\cardiac_complications_cleaned.csv
 - C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\demography_cleaned.csv
 - C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\hospitalization_discharge_cleaned.csv
 - C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\labs_cleaned.csv
 - C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patient_prescriptions_cleaned.csv
 - C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patienthistory_cleaned.csv
 - C:\Users\arjai\Documents\Shraddha Job Sea

In [208]:
# One-row-per-patient tables (everything except prescriptions)
patient_level = {
    f: df for f, df in cleaned_dfs.items()
    if "prescriptions" not in f
}

for f, df in patient_level.items():
    print(f"{f}: shape={df.shape}, unique_ids={df['inpatient_number'].nunique()}, "
          f"duplicate_ids={df['inpatient_number'].duplicated().sum()}")

C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\cardiac_complications_cleaned.csv: shape=(2008, 14), unique_ids=2008, duplicate_ids=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\demography_cleaned.csv: shape=(2008, 8), unique_ids=2008, duplicate_ids=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\hospitalization_discharge_cleaned.csv: shape=(2008, 24), unique_ids=2008, duplicate_ids=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\labs_cleaned.csv: shape=(2008, 107), unique_ids=2008, duplicate_ids=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patienthistory_cleaned.csv: shape=(2008, 17), unique_ids=200

In [209]:
# Master patient ID set — every one-row-per-patient table should match this exactly
master_ids = set(next(iter(patient_level.values()))["inpatient_number"])

for f, df in patient_level.items():
    ids = set(df["inpatient_number"])
    missing = master_ids - ids
    extra = ids - master_ids
    print(f"{f}: missing_vs_master={len(missing)}, extra_vs_master={len(extra)}")
    if missing:
        print("   missing IDs:", missing)
    if extra:
        print("   extra IDs:", extra)

C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\cardiac_complications_cleaned.csv: missing_vs_master=0, extra_vs_master=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\demography_cleaned.csv: missing_vs_master=0, extra_vs_master=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\hospitalization_discharge_cleaned.csv: missing_vs_master=0, extra_vs_master=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\labs_cleaned.csv: missing_vs_master=0, extra_vs_master=0
C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patienthistory_cleaned.csv: missing_vs_master=0, extra_vs_master=0
C:\Users\arjai\Documents\Shraddha Job Sea

In [210]:
# Prescriptions is one-to-many — check membership, not exact match
prescriptions_file = [f for f in cleaned_files if "prescriptions" in f][0]
prescriptions_df = cleaned_dfs[prescriptions_file]

presc_ids = set(prescriptions_df["inpatient_number"])
missing_from_prescriptions = master_ids - presc_ids
extra_in_prescriptions = presc_ids - master_ids

print("Patients with no prescription record:", missing_from_prescriptions)
print("Prescription patient IDs not in master (should be empty):", extra_in_prescriptions)

Patients with no prescription record: {789308}
Prescription patient IDs not in master (should be empty): set()


**Expected result:** every one-row-per-patient table shows 2,008 unique IDs, 0 duplicates, 
and 0 missing/extra IDs vs. the master. Prescriptions should show exactly one missing 
patient (789308) and zero extras. If anything comes back different from this, stop and 
flag it before we touch the merge decision.

## Master Patient Table (built once, used everywhere below)

This cell builds `patient_master` — one row per patient (2,008 rows), combining 
Demography, Cardiac Complications, Hospitalization/Discharge, Labs, Patient History, 
and Responsiveness, plus derived features from Patient Prescriptions (total drug count 
and a flag per drug).

The Overall/Cross-Dataset Validation performed earlier confirmed all 6 patient-level 
tables share the identical 2,008 patient IDs, with zero missing or extra IDs in any 
table. Given that, the merge below anchors on Hospitalization/Discharge as the defined 
base population and left-joins the remaining tables onto it — this is a deliberate, 
documented choice of base population rather than an arbitrary one, even though the 
join result is mathematically identical to an inner join given the validated ID overlap.

**This is built once, right after setup.** Every Descriptive, Prescriptive, and 
Predictive question below should use `patient_master` directly — do not re-load or 
re-merge the individual cleaned files again in this notebook.

The raw long-format `patient_prescriptions_cleaned.csv` is also loaded separately as 
`prescriptions_long`, for any question that needs drug-level detail (e.g. "which 
specific diuretic") rather than a per-patient summary. `prescriptions_long` and the 
core columns of `patient_master` are frozen — new questions may add new columns, but 
should not edit these.

In [211]:
cleaned_files = {
    "demography": CLEANED_PATH / "demography_cleaned.csv",
    "cardiac": CLEANED_PATH / "cardiac_complications_cleaned.csv",
    "hospitalization": CLEANED_PATH / "hospitalization_discharge_cleaned.csv",
    "labs": CLEANED_PATH / "labs_cleaned.csv",
    "patienthistory": CLEANED_PATH / "patienthistory_cleaned.csv",
    "responsiveness": CLEANED_PATH / "responsivenes_cleaned.csv",
}

patient_level_dfs = {
    name: pd.read_csv(path)
    for name, path in cleaned_files.items()
}

for name, df in patient_level_dfs.items():
    print(f"{name}: {df.shape}")

demography: (2008, 8)
cardiac: (2008, 14)
hospitalization: (2008, 24)
labs: (2008, 107)
patienthistory: (2008, 17)
responsiveness: (2008, 7)


In [212]:
# Left join from Hospitalization as the defined base population.
# (Already confirmed via Overall/Cross-Dataset Validation: all 6 tables share the
# identical 2,008 patient IDs, so left/inner/outer are equivalent here — left is used
# for clarity on which table anchors the merge.)

patient_master = patient_level_dfs["hospitalization"]

for name, df in patient_level_dfs.items():
    if name == "hospitalization":
        continue
    patient_master = patient_master.merge(df, on="inpatient_number", how="left")

print("patient_master shape after 6-table join:", patient_master.shape)
assert patient_master.shape[0] == 2008
assert patient_master["inpatient_number"].duplicated().sum() == 0
assert patient_master.drop(columns=["inpatient_number"]).columns.is_unique

patient_master shape after 6-table join: (2008, 172)


In [213]:
# Prescriptions — kept as long format for drug-level questions...
prescriptions_long = pd.read_csv(CLEANED_PATH / "patient_prescriptions_cleaned.csv")

# ...and summarized into per-patient features for the master table
drug_count = (
    prescriptions_long.groupby("inpatient_number")
    .size()
    .rename("total_drug_count")
)

drug_flags = (
    pd.crosstab(prescriptions_long["inpatient_number"], prescriptions_long["drug_name"])
    .clip(upper=1)
    .add_prefix("drug_")
)

prescriptions_summary = drug_count.to_frame().join(drug_flags, how="outer").reset_index()

print("prescriptions_summary shape:", prescriptions_summary.shape)
print("Columns:", prescriptions_summary.columns.tolist())

prescriptions_summary shape: (2007, 27)
Columns: ['inpatient_number', 'total_drug_count', 'drug_Aspirin enteric-coated tablet', 'drug_Atorvastatin calcium tablet', 'drug_Benazepril hydrochloride tablet', 'drug_Clopidogrel Hydrogen Sulphate tablet', 'drug_Deslanoside injection', 'drug_Digoxin tablet', 'drug_Dobutamine hydrochloride injection', 'drug_Enoxaparin Sodium injection', 'drug_Furosemide injection', 'drug_Furosemide tablet', 'drug_Heparin Sodium injection', 'drug_Hydrochlorothiazide tablet', 'drug_Isoprenaline Hydrochloride injection', 'drug_Isosorbide Mononitrate Sustained Release tablet', 'drug_Meglumine Adenosine Cyclophosphate for injection', 'drug_Metoprolol Succinate Sustained-release tablet', 'drug_Metoprolol Tartrate Injection', 'drug_Milrinone injection', 'drug_Nitroglycerin injection', 'drug_Shenfu injection', 'drug_Spironolactone tablet', 'drug_Sulfotanshinone Sodium Injection', 'drug_Torasemide tablet', 'drug_Valsartan Dispersible tablet', 'drug_Warfarin Sodium Table

In [214]:
patient_master = patient_master.merge(prescriptions_summary, on="inpatient_number", how="left")

# Patient 789308 has no prescription record — fill their drug columns with 0, not NaN
drug_columns = ["total_drug_count"] + [c for c in patient_master.columns if c.startswith("drug_")]
patient_master[drug_columns] = patient_master[drug_columns].fillna(0)

print("Final patient_master shape:", patient_master.shape)
assert patient_master.shape[0] == 2008
assert patient_master["inpatient_number"].duplicated().sum() == 0

# Sanity check on the patient with no prescriptions
display(patient_master.loc[patient_master["inpatient_number"] == 789308, drug_columns])

Final patient_master shape: (2008, 198)


,total_drug_count,drug_Aspirin enteric-coated tablet,drug_Atorvastatin calcium tablet,drug_Benazepril hydrochloride tablet,drug_Clopidogrel Hydrogen Sulphate tablet,drug_Deslanoside injection,drug_Digoxin tablet,drug_Dobutamine hydrochloride injection,drug_Enoxaparin Sodium injection,drug_Furosemide injection,drug_Furosemide tablet,drug_Heparin Sodium injection,drug_Hydrochlorothiazide tablet,drug_Isoprenaline Hydrochloride injection,drug_Isosorbide Mononitrate Sustained Release tablet,drug_Meglumine Adenosine Cyclophosphate for injection,drug_Metoprolol Succinate Sustained-release tablet,drug_Metoprolol Tartrate Injection,drug_Milrinone injection,drug_Nitroglycerin injection,drug_Shenfu injection,drug_Spironolactone tablet,drug_Sulfotanshinone Sodium Injection,drug_Torasemide tablet,drug_Valsartan Dispersible tablet,drug_Warfarin Sodium Tablet
1461,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [215]:
patient_master.to_csv(CLEANED_PATH / "patient_master.csv", index=False)
print("Saved:", CLEANED_PATH / "patient_master.csv")
print("Shape:", patient_master.shape)

Saved: C:\Users\arjai\Documents\Shraddha Job Search\Data Science\Python\Hackathon Sept 2026\Python_Hackathon_Sep_2026\cleaned_data\patient_master.csv
Shape: (2008, 198)


In [216]:
patient_master = pd.read_csv(CLEANED_PATH / "patient_master.csv")
prescriptions_long = pd.read_csv(CLEANED_PATH / "patient_prescriptions_cleaned.csv")

print("patient_master:", patient_master.shape)
print("prescriptions_long:", prescriptions_long.shape)

patient_master: (2008, 198)
prescriptions_long: (15362, 2)
